# Next-day (T+1) spread prediction: unused anchor-time information in the TRACE tape

**Companion to `next_day_received_5.ipynb`.** That notebook concluded that the only anchor-time signal that matters is the target-side CPP quote (`cpp_side`, shrunk by $\beta \approx 0.67$), that the 82-feature LightGBM adds little, and that the remaining gap to the target-print intraday oracle (2.31 vs 1.18 bps) is "mysterious" beyond the ~30% explained by realised open bumps.

This notebook tests a different reading of the same result: **the 0.67 is an average over prints of very different informativeness, and the tape tells you which is which.** It does four things:

1. **Reframes the gap as CPP drift.** For every anchor-target pair,
   $\text{TARGET} = \underbrace{q^{CPP}_{\ell_u}(t)-s_t}_{\texttt{cpp\_side}} + \underbrace{q^{CPP}_{\ell_u}(u)-q^{CPP}_{\ell_u}(t)}_{\text{CPP drift}} + \underbrace{s_u-q^{CPP}_{\ell_u}(u)}_{\text{target-print noise}}$.
   The oracle is the model with the drift term known. Section 4 measures the three terms and splits the drift into rest-of-day-$T$, overnight and day-$T{+}1$ pieces, so the "70%" gets a location.
2. **Label hygiene (Section 1).** In the sample tape, ~37% of `TRADE_TYPE == 'D'` prints are customer trades (`CONTRA_PARTY_TYPE == 'C'`) that sit $\pm$1.2 bps from CPP mid like B/S prints; ~40% of prints occur in same-second bursts of $\ge$20 CUSIPs (portfolio trades and interdealer sessions); 9.5% of all prints land in the single minute 16:00. None of this is visible to `TRADE_TYPE`.
3. **New anchor-time feature blocks (Section 2)** that are economically about *how much to trust the anchor print and the CPP quote*, and that live on a one-day rather than one-hour time scale:
   - **SIDE**: effective side from `SIDE`+`CONTRA_PARTY_TYPE`, riskless-principal pairs, burst / session / EOD flags, `RENUMERATION_INDICATOR`, `ATS_INDICATOR`.
   - **CPPBIAS**: history of print-vs-CPP deviation at bond / issuer / market level (CPP has a persistent bias per bond and lags the market intraday), CPP momentum, interdealer-session snapshots.
   - **MULTIDAY**: 5d/20d signed flow (dealer inventory), block recency, multi-day relative value on I-spread, issuer-curve residual, liquidity, new-issue age.
   - **TIME**: minutes to close, morning anchor, target-day weekday / month-end regimes.
4. **Walk-forward evaluation (Section 5)** with the same folds, target-equal MAE and paired-day $t$ as the companion notebook: a ladder from `dummy` to a *print-quality-aware* $\beta$ rule, a small LAD model, LightGBM on a compact reproduction of the old core features, LightGBM with each new block added (ablation), a validation-to-test residual test, permutation importance by block, and a two-stage model that predicts CPP drift directly.

Everything is fitted on train dates only and scored on test dates; calibration layers use validation dates only.

**How to run.** Same working directory and data layout as the companion notebook (`ROOT = Path.cwd()`, cached tape at `artifacts/experiments/next_day/data_ig_merged.parquet` + `bm_dict_merged.pkl`; falls back to the pipeline CSVs and `bond_pricer.data.load_merged_prints`). The cache must contain the raw TRACE flags (`SIDE`, `CONTRA_PARTY_TYPE`, `RENUMERATION_INDICATOR`, `ATS_INDICATOR`) and `I_SPREAD`; the load cell lists anything missing and the dependent features are skipped. Runtime knobs are in the first code cell: `RUN_LGBM`, `RUN_ABLATION`, `RUN_RESIDUAL_TEST`, `RUN_PERMUTATION`, `RUN_DRIFT_MODEL`, and `LGBM_TRAIN_ROW_CAP` (default 3M training pairs per fold). With everything on, expect roughly 2 LightGBM fits per fold for the ladder plus 4 for the ablation, 1 for the drift model and 2 small residual models; on the 3.4M-print tape that is a few hours, comparable to the companion's Section 5.4 reruns. Tables and figures are written to `artifacts/experiments/next_day_anchor_features/`. When the tape is small (< 100k prints) the notebook switches to `SAMPLE_MODE` (fewer, shorter folds; lower burst thresholds) so every code path is exercised; the statistics in that mode are not meaningful.

In [ ]:
from pathlib import Path
import sys, pickle, warnings, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 120); pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid", context="notebook", font_scale=0.9)

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# ---- same raw inputs / cache layout as next_day_received_5.ipynb -------------------------------------------
pipeline_csv_1 = ROOT / "data/raw/pipeline/2026-08-06/data_pipeline.csv_20260806"
benchmark_csv_1 = ROOT / "data/raw/pipeline/2026-08-06/DailyCloseUSTBenchmarks.csv_20260806"
pipeline_csv_2 = ROOT / "data/raw/pipeline/2026-05-06/data_pipeline.csv_20260506"
benchmark_csv_2 = ROOT / "data/raw/pipeline/2026-05-06/DailyCloseUSTBenchmarks.csv_20260506"
pipeline_csv_3 = ROOT / "data/raw/pipeline/2026-02-06/data_pipeline.csv_20260206"
benchmark_csv_3 = ROOT / "data/raw/pipeline/2026-02-06/DailyCloseUSTBenchmarks.csv_20260206"
PIPELINE_INPUTS = [(pipeline_csv_1, benchmark_csv_1), (pipeline_csv_2, benchmark_csv_2), (pipeline_csv_3, benchmark_csv_3)]
liquid_trade_count_threshold, qav_threshold_ig, qav_threshold_hy = 15, 0.005, 0.2

RAW_CACHE_DIR = ROOT / "artifacts" / "experiments" / "next_day"
RAW_CACHE_DIR.mkdir(parents=True, exist_ok=True)
DATA_IG_CACHE = RAW_CACHE_DIR / "data_ig_merged.parquet"
BM_DICT_CACHE = RAW_CACHE_DIR / "bm_dict_merged.pkl"
OUT_DIR = ROOT / "artifacts" / "experiments" / "next_day_anchor_features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- runtime knobs ------------------------------------------------------------------------------------------
RUN_LGBM = True            # LightGBM rungs (old core, old core + all new blocks)
RUN_ABLATION = True        # one LightGBM per new block added to the old core (4 extra fits per fold)
RUN_RESIDUAL_TEST = True   # validation-trained residual model on new features, applied to test
RUN_PERMUTATION = True     # permutation importance of the full model by block
RUN_DRIFT_MODEL = True     # two-stage model: predict CPP drift, add to cpp_side_eff
LGBM_TRAIN_ROW_CAP = 3_000_000   # random cap on training pairs per fold (None = all)
PERM_ROWS = 100_000
PLOT_DAYS = 6              # number of days shown in intraday plots
SEED = 20260922

def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

def bps(x):
    if isinstance(x, pd.DataFrame):
        return x.apply(pd.to_numeric, errors="coerce") * 100.0
    return pd.to_numeric(x, errors="coerce") * 100.0

t_start = time.time()
print(f"ROOT = {ROOT}\nfigures and tables -> {OUT_DIR}")

## 0. Load the tape

Identical to Section 1 of the companion notebook: cached merged IG tape if present, otherwise the three pipeline snapshots are merged and run through `bond_pricer.data.load_merged_prints`. Prints are de-duplicated on (CUSIP, timestamp, quantity, trade type) and IG sizes are un-capped with `ESTIMATED_QUANTITY`. The one difference is that this notebook needs the raw TRACE flags (`SIDE`, `CONTRA_PARTY_TYPE`, `RENUMERATION_INDICATOR`, `ATS_INDICATOR`) and a few extra pipeline columns (`I_SPREAD`, `TRADE_COUNTS_PREV_MONTH`, `IS_144A`); the cell reports which are missing and degrades gracefully.

In [ ]:
def build_benchmark_prev_close_table(pipeline_inputs):
    dfs = []
    for inp in pipeline_inputs:
        d = pd.read_csv(Path(inp[1]), index_col=0).T.drop_duplicates()
        d.columns = d.columns.str.strip()
        d.index = pd.to_datetime(d.index)
        dfs.append(d)
    table = {}
    for frame in sorted(dfs, key=lambda f: f.index.max()):
        frame = frame.sort_index(kind="stable")
        frame = frame[~frame.index.duplicated(keep="last")]
        frame = frame.loc[:, ~frame.columns.duplicated(keep=False)]
        table.update(frame.shift(1).unstack().dropna().to_dict())
    return table

def to_ny_datetime(series):
    values = pd.to_datetime(series, errors="coerce")
    if getattr(values.dt, "tz", None) is None:
        return values.astype("datetime64[ns]").dt.tz_localize("America/New_York")
    return values.dt.tz_convert("America/New_York")

def merge_pipeline_snapshots(pipeline_inputs, out_dir=ROOT / "data" / "raw" / "pipeline" / "_merged"):
    out_dir.mkdir(parents=True, exist_ok=True)
    pipeline_csv, benchmark_csv = out_dir / "data_pipeline.csv", out_dir / "DailyCloseUSTBenchmarks.csv"
    frames, claimed = [], set()
    for pcsv, _ in pipeline_inputs:
        raw = pd.read_csv(pcsv, low_memory=False)
        stamp = pd.to_datetime(raw["EFFECTIVE_DATETIME_TS"], errors="coerce")
        raw["EFFECTIVE_DATETIME_TS"] = stamp
        day = stamp.dt.normalize()
        keep = ~day.isin(claimed)
        claimed |= set(day[keep].unique())
        frames.append(raw[keep])
    raw = pd.concat(frames, ignore_index=True).sort_values("EFFECTIVE_DATETIME_TS", kind="stable", ignore_index=True)
    raw.to_csv(pipeline_csv, index=False)
    tables = []
    for _, bcsv in pipeline_inputs:
        t = pd.read_csv(bcsv, index_col=0).T
        t.columns = t.columns.str.strip(); t.index = pd.to_datetime(t.index)
        tables.append(t)
    bm = pd.concat(tables); bm = bm[~bm.index.duplicated(keep="first")].sort_index()
    bm.T.to_csv(benchmark_csv)
    return pipeline_csv, benchmark_csv

if DATA_IG_CACHE.exists() and BM_DICT_CACHE.exists():
    print(f"loading cached IG tape: {DATA_IG_CACHE}")
    data_ig = pd.read_parquet(DATA_IG_CACHE)
    with BM_DICT_CACHE.open("rb") as h:
        bm_dict = pickle.load(h)
else:
    from bond_pricer.data import load_merged_prints
    merged_pipeline_csv, merged_benchmark_csv = merge_pipeline_snapshots(PIPELINE_INPUTS)
    data_ig = load_merged_prints(pipeline_csv=merged_pipeline_csv, benchmark_csv=merged_benchmark_csv,
                                 liquid_trade_count_threshold=liquid_trade_count_threshold,
                                 qav_threshold_ig=qav_threshold_ig, qav_threshold_hy=qav_threshold_hy)
    data_ig["EFFECTIVE_DATETIME_TS"] = to_ny_datetime(data_ig["EFFECTIVE_DATETIME_TS"])
    bm_dict = build_benchmark_prev_close_table([(merged_pipeline_csv, merged_benchmark_csv)])
    data_ig.to_parquet(DATA_IG_CACHE, index=False)
    with BM_DICT_CACHE.open("wb") as h:
        pickle.dump(bm_dict, h)

data_ig["EFFECTIVE_DATETIME_TS"] = to_ny_datetime(data_ig["EFFECTIVE_DATETIME_TS"])
data_ig["DATE"] = pd.to_datetime(data_ig["DATE"]).dt.normalize()

_k = ["CUSIP", "EFFECTIVE_DATETIME_TS", "QUANTITY", "TRADE_TYPE"]
_n0 = len(data_ig)
data_ig = data_ig.drop_duplicates(_k, keep="first").sort_values("EFFECTIVE_DATETIME_TS", kind="stable").reset_index(drop=True)
print(f"dropped {_n0 - len(data_ig):,} duplicate prints -> {len(data_ig):,} remain")
_est = pd.to_numeric(data_ig.get("ESTIMATED_QUANTITY"), errors="coerce").fillna(0.0)
data_ig["IS_SIZE_ESTIMATED"] = _est.gt(0).astype("int8")
data_ig["QUANTITY"] = np.maximum(pd.to_numeric(data_ig["QUANTITY"], errors="coerce"), _est)
if "TRADE_ROW_ID" not in data_ig.columns:
    data_ig["TRADE_ROW_ID"] = np.arange(len(data_ig), dtype="int64")

REQUIRED = ["CUSIP", "ISSUER", "EFFECTIVE_DATETIME_TS", "DATE", "BM_CUSIP", "BM_SPREAD", "TRADE_TYPE", "QUANTITY",
            "BID_SPREAD_CPP", "ASK_SPREAD_CPP", "MID_SPREAD_CPP", "YRS_TO_MATURITY", "BM_TENOR_GROUP"]
OPTIONAL = ["SIDE", "CONTRA_PARTY_TYPE", "RENUMERATION_INDICATOR", "ATS_INDICATOR", "REPORTING_PARTY_TYPE",
            "I_SPREAD", "TRADE_COUNTS_PREV_MONTH", "IF_LIQUID_BOND", "IS_144A", "COUPON", "PRICE", "CDX_TRADE",
            "MOST_RECENT_TQW_SPREAD", "MOST_RECENT_EXEC_TIME", "TQW_SPREAD", "PREV_TRADE_TYPE", "PREV_QUANTITY",
            "PREV_BM_SPREAD", "D_BM_SPREAD", "BM_SPREAD_DEV", "ROLLING_BM_SPREAD", "MEAN_ISSUER_SPREAD_DEV",
            "NUM_OF_ISSUER_TRADES_SINCE_PREV", "BM_SPREAD_STD_GROUP_BY_TYPE", "BM_YIELD_STD", "D_CDX_TRADE", "CDX_TRADE_DEV"]
_missing_req = [c for c in REQUIRED if c not in data_ig.columns]
_missing_opt = [c for c in OPTIONAL if c not in data_ig.columns]
assert not _missing_req, f"missing required columns: {_missing_req}"
HAS = {c: (c in data_ig.columns) for c in OPTIONAL}
print("missing optional columns (features depending on them are skipped):", _missing_opt or "none")

n_prints = len(data_ig)
SAMPLE_MODE = n_prints < 100_000
BURST_MIN_CUSIPS = 20 if not SAMPLE_MODE else 3
SESSION_MIN_CUSIPS = 100 if not SAMPLE_MODE else 5
print(f"{n_prints:,} prints, {data_ig['CUSIP'].nunique():,} CUSIPs, {data_ig['DATE'].nunique()} days "
      f"({data_ig['DATE'].min():%Y-%m-%d} -> {data_ig['DATE'].max():%Y-%m-%d}); SAMPLE_MODE={SAMPLE_MODE} "
      f"(burst >= {BURST_MIN_CUSIPS} CUSIPs/sec, session >= {SESSION_MIN_CUSIPS} CUSIPs/sec)")

## 1. EDA: what the tape says that `TRADE_TYPE` does not

### 1.1 Effective side: `TRADE_TYPE == 'D'` is not interdealer

`TRADE_TYPE` is the side label the whole pipeline (intraday model, companion notebook) uses: B = dealer buys from customer (print sits near the CPP bid, i.e. wide), S = dealer sells (near the ask, tight), D = interdealer (near mid). The raw TRACE fields `SIDE` (reporting dealer's side) and `CONTRA_PARTY_TYPE` (C customer, D dealer, A ATS, T affiliate/other) allow a second classification:

$$
\texttt{EFF\_SIDE} = \begin{cases} D & \text{CONTRA} \in \{D, T\} \\ B & \text{otherwise, SIDE} = B \\ S & \text{otherwise, SIDE} = S \end{cases}
$$

The cell cross-tabulates the two and shows the deviation of each group from the CPP mid, $s_t - m^{CPP}_t$ in bps. If a `TRADE_TYPE = D` group with a customer contra sits at $\pm$1 bps like the B/S groups, the D label is hiding sided prints and every side-dependent feature (`CPP_TARGET_SIDE_DEV`, `SIDE_FLIP`, `EXPECTED_BOUNCE`, `SIDE_PAIR_PRIOR`, and the target-side scenario itself) is wrong for them.

In [ ]:
BASE_COLS = [c for c in ["TRADE_ROW_ID", "CUSIP", "ISSUER", "DATE", "EFFECTIVE_DATETIME_TS", "BM_CUSIP", "BM_TENOR_GROUP",
                         "BM_SPREAD", "BID_SPREAD_CPP", "ASK_SPREAD_CPP", "MID_SPREAD_CPP", "TRADE_TYPE", "QUANTITY",
                         "YRS_TO_MATURITY", "IS_SIZE_ESTIMATED", *OPTIONAL] if c in data_ig.columns]
df = data_ig[BASE_COLS].copy()
for c in ["BM_SPREAD", "BID_SPREAD_CPP", "ASK_SPREAD_CPP", "MID_SPREAD_CPP", "QUANTITY", "YRS_TO_MATURITY"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["TRADE_TYPE"] = df["TRADE_TYPE"].astype(str)

# ---- effective side ---------------------------------------------------------------------------------------------
if HAS["SIDE"] and HAS["CONTRA_PARTY_TYPE"]:
    _contra = df["CONTRA_PARTY_TYPE"].astype(str); _side = df["SIDE"].astype(str)
    df["EFF_SIDE"] = np.where(_contra.isin(["D", "T"]), "D", np.where(_side.eq("B"), "B", np.where(_side.eq("S"), "S", df["TRADE_TYPE"])))
else:
    print("WARNING: SIDE / CONTRA_PARTY_TYPE missing -> EFF_SIDE falls back to TRADE_TYPE; the SIDE block is inert.")
    df["EFF_SIDE"] = df["TRADE_TYPE"]
df["IS_TRUE_DD"] = df["EFF_SIDE"].eq("D").astype("int8")
df["IS_MISLABELED_D"] = (df["TRADE_TYPE"].eq("D") & df["EFF_SIDE"].ne("D")).astype("int8")
df["IS_MARKUP"] = df["RENUMERATION_INDICATOR"].astype(str).eq("M").astype("int8") if HAS["RENUMERATION_INDICATOR"] else np.int8(0)
df["IS_ATS"] = df["ATS_INDICATOR"].astype(str).eq("Y").astype("int8") if HAS["ATS_INDICATOR"] else np.int8(0)
df["IS_144A_FLAG"] = df["IS_144A"].astype(str).eq("Y").astype("int8") if HAS["IS_144A"] else np.int8(0)

# ---- quotes and deviations (spread units: 0.01 = 1 bp, as in the companion notebook) -----------------------------
df["CPP_WIDTH"] = (df["BID_SPREAD_CPP"] - df["ASK_SPREAD_CPP"]).abs().replace(0.0, np.nan)
def side_quote(side):
    side = side.astype(str)
    return np.select([side.eq("B").to_numpy(), side.eq("S").to_numpy()],
                     [df["BID_SPREAD_CPP"].to_numpy(), df["ASK_SPREAD_CPP"].to_numpy()], df["MID_SPREAD_CPP"].to_numpy())
df["Q_TT"] = side_quote(df["TRADE_TYPE"]); df["Q_EFF"] = side_quote(df["EFF_SIDE"])
df["DEV_MID"] = df["BM_SPREAD"] - df["MID_SPREAD_CPP"]     # print minus CPP mid
df["DEV_TT"] = df["BM_SPREAD"] - df["Q_TT"]                 # print minus its TRADE_TYPE-side quote
df["DEV_EFF"] = df["BM_SPREAD"] - df["Q_EFF"]               # print minus its EFF_SIDE quote

_grp = [c for c in ["TRADE_TYPE", "SIDE", "CONTRA_PARTY_TYPE", "RENUMERATION_INDICATOR"] if c in df.columns]
xt = (df.groupby(_grp, observed=True)
        .agg(n=("DEV_MID", "size"), share_pct=("DEV_MID", lambda s: 100 * len(s) / len(df)),
             dev_mid_mean_bps=("DEV_MID", lambda s: 100 * s.mean()), dev_mid_median_bps=("DEV_MID", lambda s: 100 * s.median()),
             dev_mid_mad_bps=("DEV_MID", lambda s: 100 * (s - s.median()).abs().median()),
             qty_median=("QUANTITY", "median"), eff_side=("EFF_SIDE", lambda s: s.mode().iloc[0])))
print("TRADE_TYPE x raw TRACE flags -> print minus CPP mid (bps):")
print(xt.round(2).to_string())
print(f"\nshare of TRADE_TYPE == 'D' prints that are customer trades by EFF_SIDE: "
      f"{df.loc[df.TRADE_TYPE.eq('D'), 'IS_MISLABELED_D'].mean():.1%}  ({df['IS_MISLABELED_D'].sum():,} prints, {df['IS_MISLABELED_D'].mean():.1%} of all)")
xt.to_csv(OUT_DIR / "eda_side_crosstab.csv")

# ---- same-time check: does referencing the EFF_SIDE quote reduce |print - quote|? ----------------------------------
rows = []
for lab, m in [("all prints", np.ones(len(df), bool)), ("TRADE_TYPE == D", df.TRADE_TYPE.eq("D").to_numpy()),
               ("TRADE_TYPE == D & customer contra", df.IS_MISLABELED_D.eq(1).to_numpy()), ("TRADE_TYPE in B/S", df.TRADE_TYPE.isin(["B", "S"]).to_numpy())]:
    d = df[m]
    rows.append({"subset": lab, "n": len(d), "MAE_TRADE_TYPE_quote_bps": 100 * d.DEV_TT.abs().mean(), "MAE_EFF_SIDE_quote_bps": 100 * d.DEV_EFF.abs().mean(),
                 "gain_%": 100 * (1 - d.DEV_EFF.abs().mean() / d.DEV_TT.abs().mean()),
                 "medAE_TRADE_TYPE": 100 * d.DEV_TT.abs().median(), "medAE_EFF_SIDE": 100 * d.DEV_EFF.abs().median()})
side_check = pd.DataFrame(rows).set_index("subset")
print("\nsame-time |print - side quote| using TRADE_TYPE vs EFF_SIDE to pick the quote side:")
print(side_check.round(3).to_string())

# ---- plot ---------------------------------------------------------------------------------------------------------
_pl = df.assign(dev=bps(df.DEV_MID).clip(-6, 6), group=df.TRADE_TYPE + " / eff " + df.EFF_SIDE)
_order = sorted(_pl.group.unique())
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [3, 2]})
sns.violinplot(data=_pl.sample(min(len(_pl), 300_000), random_state=SEED), x="group", y="dev", order=_order, ax=axes[0], cut=0, inner="quartile", density_norm="width")
axes[0].axhline(0, color="k", lw=0.8); axes[0].set_title("print minus CPP mid by TRADE_TYPE / EFF_SIDE (bps, clipped to +-6)"); axes[0].set_xlabel(""); axes[0].tick_params(axis="x", rotation=30)
_bar = side_check[["MAE_TRADE_TYPE_quote_bps", "MAE_EFF_SIDE_quote_bps"]]
_bar.plot(kind="bar", ax=axes[1]); axes[1].set_title("same-time |print - side quote|, bps"); axes[1].set_xlabel(""); axes[1].tick_params(axis="x", rotation=20)
for i, v in enumerate(side_check["gain_%"]):
    axes[1].annotate(f"{v:+.1f}%", (i, _bar.iloc[i].max()), ha="center", va="bottom", fontsize=9)
savefig("eda_1_1_effective_side")

### 1.2 Riskless-principal pairs and same-second bursts

Two more things a single print's label cannot show:

- **Riskless-principal pairs.** A dealer fills a customer and immediately crosses the position with another dealer: two prints for the same CUSIP at the same second and the same quantity, one customer leg and one interdealer leg, at the same price. The interdealer leg is *not* a mid print; it is at the customer's price. `IS_RP_PAIR` flags both legs, `IS_RP_D_LEG` the interdealer leg.
- **Bursts.** Portfolio trades and interdealer matching sessions print hundreds of CUSIPs in the same second. For each print, `N_CUSIP_SEC` counts distinct CUSIPs printing in that second market-wide, `BURST_PURITY` the share of the dominant effective side in that second, `IS_BURST` marks $\ge$ `BURST_MIN_CUSIPS`, `IS_SESSION` marks large ($\ge$ `SESSION_MIN_CUSIPS`) and $\ge$90% interdealer bursts, `IS_EOD` marks 16:00-16:01. `N_PRINTS_TRAIL_60S` is the strictly-past version (prints in the previous 60 seconds, any CUSIP) for readers who do not want to use the same-second count.

Why it matters for T+1: a portfolio-trade line is priced at the basket level, so its deviation from CPP is not bond information and should revert almost fully; interdealer sessions are the cleanest cross-sectional snapshot of fair value the tape offers; and the 16:00 burst is the largest anchor bucket in the companion notebook and the one where the LightGBM loses to `cpp_side`.

In [ ]:
df = df.sort_values(["EFFECTIVE_DATETIME_TS", "CUSIP", "TRADE_ROW_ID"], kind="stable").reset_index(drop=True)
ts = df["EFFECTIVE_DATETIME_TS"]
df["SEC"] = ts.dt.floor("s")
df["HOUR_F"] = ts.dt.hour + ts.dt.minute / 60.0
df["MINUTES_TO_CLOSE"] = (16.0 - df["HOUR_F"]) * 60.0
df["IS_EOD"] = (ts.dt.hour.eq(16) & ts.dt.minute.le(1)).astype("int8")

# ---- riskless-principal pairs ----------------------------------------------------------------------------------------
_rp_key = ["CUSIP", "SEC", "QUANTITY"]
_g = df.groupby(_rp_key, sort=False)
_n_same = _g["EFF_SIDE"].transform("size")
_has_d = _g["IS_TRUE_DD"].transform("max"); _has_c = _g["IS_TRUE_DD"].transform("min").eq(0)
df["IS_RP_PAIR"] = (_n_same.ge(2) & _has_d.eq(1) & _has_c).astype("int8")
df["IS_RP_D_LEG"] = (df["IS_RP_PAIR"].eq(1) & df["IS_TRUE_DD"].eq(1)).astype("int8")

# ---- bursts ----------------------------------------------------------------------------------------------------------
_gs = df.groupby("SEC", sort=False)
df["N_SEC_ALL"] = _gs["CUSIP"].transform("size").astype("int32")
df["N_CUSIP_SEC"] = _gs["CUSIP"].transform("nunique").astype("int32")
df["N_PRINTS_SAME_SEC_CUSIP"] = df.groupby(["SEC", "CUSIP"], sort=False)["CUSIP"].transform("size").astype("int16")
_cnt_side = df.groupby(["SEC", "EFF_SIDE"], sort=False)["CUSIP"].transform("size")
df["BURST_PURITY"] = (_cnt_side.groupby(df["SEC"]).transform("max") / df["N_SEC_ALL"]).astype("float32")
_share_d = _gs["IS_TRUE_DD"].transform("mean")
df["IS_BURST"] = df["N_CUSIP_SEC"].ge(BURST_MIN_CUSIPS).astype("int8")
df["IS_SESSION"] = (df["N_CUSIP_SEC"].ge(SESSION_MIN_CUSIPS) & _share_d.ge(0.9)).astype("int8")
df["IS_PT_BURST"] = (df["IS_BURST"].eq(1) & df["IS_SESSION"].eq(0)).astype("int8")   # customer portfolio-trade-like burst
# strictly-past activity: prints in the previous 60 s (any CUSIP)
_t_ns = ts.astype("int64").to_numpy()
_hi = np.searchsorted(_t_ns, _t_ns, side="left"); _lo = np.searchsorted(_t_ns, _t_ns - 60 * 10**9, side="left")
df["N_PRINTS_TRAIL_60S"] = (_hi - _lo).astype("int32")

def burst_class(d):
    return np.select([d.IS_SESSION.eq(1), d.IS_PT_BURST.eq(1) & d.IS_EOD.eq(1), d.IS_PT_BURST.eq(1), d.IS_EOD.eq(1)],
                     ["interdealer session", "EOD burst", "intraday burst", "EOD single"], "RFQ-like")
df["PRINT_CLASS"] = burst_class(df)

print(f"riskless-principal pairs: {df.IS_RP_PAIR.mean():.2%} of prints; interdealer legs in pairs: {df.IS_RP_D_LEG.sum():,} "
      f"({df.IS_RP_D_LEG.sum() / max(df.IS_TRUE_DD.sum(), 1):.1%} of interdealer prints)")
_rp = df[df.IS_RP_D_LEG.eq(1)]; _nrp = df[df.IS_TRUE_DD.eq(1) & df.IS_RP_D_LEG.eq(0)]
print(f"   interdealer leg in a pair: print - CPP mid mean {100 * _rp.DEV_MID.mean():+.2f}, |.| {100 * _rp.DEV_MID.abs().mean():.2f} bps  "
      f"vs standalone interdealer {100 * _nrp.DEV_MID.mean():+.2f}, |.| {100 * _nrp.DEV_MID.abs().mean():.2f} bps")
for thr in sorted({3, 5, 20, 50, 100, BURST_MIN_CUSIPS, SESSION_MIN_CUSIPS}):
    m = df.N_CUSIP_SEC.ge(thr)
    print(f"prints in same-second bursts of >= {thr:>3} CUSIPs: {m.mean():6.1%} of prints; mean side purity {df.loc[m, 'BURST_PURITY'].mean():.2f}")
print(f"prints in 16:00-16:01: {df.IS_EOD.mean():.1%}")

cls = (df.groupby(["PRINT_CLASS", "EFF_SIDE"], observed=True)
         .agg(n=("DEV_MID", "size"), share_pct=("DEV_MID", lambda s: 100 * len(s) / len(df)),
              dev_mid_mean_bps=("DEV_MID", lambda s: 100 * s.mean()), abs_dev_eff_mean_bps=("DEV_EFF", lambda s: 100 * s.abs().mean()),
              abs_dev_eff_median_bps=("DEV_EFF", lambda s: 100 * s.abs().median()), markup_share=("IS_MARKUP", "mean"), qty_median=("QUANTITY", "median")))
print("\nprint class x effective side:")
print(cls.round(2).to_string())
cls.to_csv(OUT_DIR / "eda_print_class.csv")

# ---- plots -------------------------------------------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
ax = axes[0, 0]
_w = df.groupby("N_CUSIP_SEC").size()
ax.bar(_w.index, 100 * _w.values / len(df), width=1.0); ax.set_xscale("log"); ax.set_xlabel("distinct CUSIPs printing in the same second"); ax.set_ylabel("% of prints")
ax.axvline(BURST_MIN_CUSIPS, color="r", ls="--", lw=1, label=f"burst >= {BURST_MIN_CUSIPS}"); ax.axvline(SESSION_MIN_CUSIPS, color="k", ls=":", lw=1, label=f"session >= {SESSION_MIN_CUSIPS}"); ax.legend()
ax.set_title("how clustered in time are prints?")
ax = axes[0, 1]
_minute = (ts.dt.hour * 60 + ts.dt.minute)
_prof = pd.crosstab(_minute, df.PRINT_CLASS, normalize=False).div(df.DATE.nunique())
_prof.index = _prof.index / 60.0
_prof.plot.area(ax=ax, stacked=True, lw=0, alpha=0.85); ax.set_xlim(7.9, 17.1); ax.set_xlabel("hour of day"); ax.set_ylabel("prints per minute per day"); ax.set_title("intraday print profile by class (avg per day)"); ax.legend(fontsize=8, loc="upper left")
ax = axes[1, 0]
_pc = df.groupby("DATE").agg(burst=("IS_BURST", "mean"), eod=("IS_EOD", "mean"), session=("IS_SESSION", "mean"), mislabeled=("IS_MISLABELED_D", "mean"))
(_pc * 100).plot(ax=ax, lw=1.2); ax.set_ylabel("% of the day's prints"); ax.set_title("daily share of burst / EOD / session / mislabeled-D prints"); ax.set_xlabel("")
ax = axes[1, 1]
_bx = df.assign(a=bps(df.DEV_EFF).abs().clip(0, 8))
sns.boxplot(data=_bx.sample(min(len(_bx), 300_000), random_state=SEED), x="PRINT_CLASS", y="a", hue="EFF_SIDE", ax=ax, showfliers=False)
ax.set_title("|print - EFF_SIDE quote| by print class (bps, clipped at 8)"); ax.set_xlabel(""); ax.set_ylabel("bps"); ax.tick_params(axis="x", rotation=15)
savefig("eda_1_2_bursts")

### 1.3 How much does the CPP composite move toward a print, and does it depend on the print?

For consecutive same-CUSIP prints $k-1 \to k$, regress the change of the CPP mid on the previous print's deviation from that mid:

$$
m^{CPP}_k - m^{CPP}_{k-1} = \alpha + \lambda\,(s_{k-1} - m^{CPP}_{k-1}) + \varepsilon .
$$

$\lambda$ is the share of a print's deviation that dealers' quotes absorb by the next print. A one-day-ahead version of $\lambda$ is what turns into $1-\beta$ in the companion notebook's `cpp_side x beta` rule ($\beta \approx 0.67$). The cell estimates $\lambda$ by print type: effective side, interdealer vs customer, markup vs no remuneration, ATS vs voice, size bucket, burst class, and gap to the next print. If $\lambda$ differs materially by type, a single $\beta$ is leaving money on the table and the flags belong in the rule. Benchmark rolls between the two prints are corrected with the previous-close yields as in the companion notebook.

In [ ]:
def bm_roll_offset(bm_curr, bm_past, dates):
    #   spread(t) - spread(past) + offset, offset = prev-close yield(curr BM) - prev-close yield(past BM); 0 when no roll
    idx_c = pd.MultiIndex.from_arrays([pd.Series(bm_curr).astype(object).to_numpy(), pd.to_datetime(dates).to_numpy()])
    idx_p = pd.MultiIndex.from_arrays([pd.Series(bm_past).astype(object).to_numpy(), pd.to_datetime(dates).to_numpy()])
    vc = pd.Series(idx_c.map(bm_dict), dtype="float64"); vp = pd.Series(idx_p.map(bm_dict), dtype="float64")
    changed = (pd.Series(bm_curr).astype(object).to_numpy() != pd.Series(bm_past).astype(object).to_numpy()) & pd.notna(bm_curr) & pd.notna(bm_past)
    return np.where(changed, (vc - vp).fillna(0.0).to_numpy(), 0.0)

# previous same-CUSIP print (time order, strictly earlier)
df = df.sort_values(["CUSIP", "EFFECTIVE_DATETIME_TS", "TRADE_ROW_ID"], kind="stable").reset_index(drop=True)
_gc = df.groupby("CUSIP", sort=False)
for c in ["EFFECTIVE_DATETIME_TS", "BM_SPREAD", "MID_SPREAD_CPP", "CPP_WIDTH", "BM_CUSIP", "DEV_EFF", "DEV_MID", "EFF_SIDE", "IS_MARKUP", "IS_ATS", "QUANTITY", "PRINT_CLASS", "IS_TRUE_DD", "DATE"]:
    df[f"P_{c}"] = _gc[c].shift(1)
df["HOURS_SINCE_PREV_TRADE"] = (df["EFFECTIVE_DATETIME_TS"] - df["P_EFFECTIVE_DATETIME_TS"]).dt.total_seconds() / 3600.0
_off = bm_roll_offset(df["BM_CUSIP"].to_numpy(), df["P_BM_CUSIP"].to_numpy(), df["DATE"].to_numpy())
df["CPP_MID_CHG_SINCE_PREV"] = (df["MID_SPREAD_CPP"] - df["P_MID_SPREAD_CPP"] + _off).astype("float32")
df["CPP_WIDTH_CHG_SINCE_PREV"] = (df["CPP_WIDTH"] - df["P_CPP_WIDTH"]).astype("float32")
df["PREV_DEV_EFF"] = df["P_DEV_EFF"].astype("float32")     # previous print's deviation from its own EFF_SIDE quote
df["SAME_DAY_PREV"] = df["P_DATE"].eq(df["DATE"])

# ---- lambda by print type -------------------------------------------------------------------------------------------------
m = df.P_DEV_MID.notna() & df.HOURS_SINCE_PREV_TRADE.gt(1 / 120) & df.HOURS_SINCE_PREV_TRADE.lt(30)
d = df.loc[m, ["CPP_MID_CHG_SINCE_PREV", "P_DEV_MID", "P_EFF_SIDE", "P_IS_TRUE_DD", "P_IS_MARKUP", "P_IS_ATS", "P_QUANTITY", "P_PRINT_CLASS", "HOURS_SINCE_PREV_TRADE"]].copy()
d["y"] = bps(d.CPP_MID_CHG_SINCE_PREV).clip(-15, 15); d["x"] = bps(d.P_DEV_MID).clip(-15, 15)
d["prev size"] = pd.cut(d.P_QUANTITY, [0, 2e5, 1e6, 4.99e6, 1e12], labels=["<=200k", "200k-1M", "1M-5M", ">=5M"]).astype(str)
d["gap to next print"] = pd.cut(d.HOURS_SINCE_PREV_TRADE, [0, 1 / 6, 1, 4, 30], labels=["<10min", "10-60min", "1-4h", "4-30h"]).astype(str)
d["prev remuneration"] = np.where(d.P_IS_MARKUP.eq(1), "markup (M)", "none / other")
d["prev venue"] = np.where(d.P_IS_ATS.eq(1), "ATS", "voice")
d["prev side"] = d.P_EFF_SIDE.astype(str).map({"B": "B dealer buys", "S": "S dealer sells", "D": "D interdealer"})
d["prev class"] = d.P_PRINT_CLASS.astype(str)

def ols_slope(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
    if len(x) < 50 or x.std() == 0: return np.nan, np.nan, len(x)
    xc, yc = x - x.mean(), y - y.mean()
    b = (xc * yc).sum() / (xc ** 2).sum()
    resid = yc - b * xc
    se = math.sqrt((resid ** 2).sum() / (len(x) - 2) / (xc ** 2).sum())
    return b, se, len(x)

rows = []
b_all, se_all, n_all = ols_slope(d.x, d.y); rows.append({"axis": "all", "level": "all", "lambda": b_all, "se": se_all, "n": n_all})
for axis in ["prev side", "prev remuneration", "prev venue", "prev size", "prev class", "gap to next print"]:
    for lvl, g in d.groupby(axis, observed=True):
        b, se, n = ols_slope(g.x, g.y)
        rows.append({"axis": axis, "level": str(lvl), "lambda": b, "se": se, "n": n})
lam = pd.DataFrame(rows)
print("lambda = share of the previous print's deviation from CPP mid that the CPP mid absorbs by the next print:")
print(lam.round(3).to_string(index=False))
lam.to_csv(OUT_DIR / "eda_cpp_response_lambda.csv", index=False)

fig, ax = plt.subplots(figsize=(13, 5.5))
_lp = lam[lam.axis != "all"].copy(); _lp["label"] = _lp.axis + ": " + _lp.level
ax.barh(_lp.label, _lp["lambda"], xerr=1.96 * _lp.se, color=sns.color_palette("tab10")[0], alpha=0.8)
ax.axvline(b_all, color="k", ls="--", lw=1, label=f"all prints {b_all:.2f}"); ax.invert_yaxis(); ax.set_xlabel("lambda (CPP mid response to previous print's deviation)"); ax.legend()
ax.set_title("dealers' composite treats prints differently: CPP response by previous-print type (95% CI)")
savefig("eda_1_3_cpp_response_by_print_type")

### 1.4 Is the print-vs-CPP deviation persistent? Bond level and market level

If part of $s_t - q^{CPP}_{\ell}(t)$ is a standing bias of the composite for that bond, consecutive prints will show correlated side-adjusted deviations, and the T+1 print will *not* revert that part. The cell reports the correlation and slope of $\texttt{DEV\_EFF}_k$ on $\texttt{DEV\_EFF}_{k-1}$ by gap bucket (winsorised at $\pm$10 bps).

At market level, if the composite lags a market move, prints will sit systematically tight (tightening day) or wide (widening day) of their side quotes for hours. The plot shows the hourly market-wide median of `DEV_EFF` for the `PLOT_DAYS` days with the largest intraday CDX range, with the CDX path on the right axis, and a scatter of the daily 9-14h mean deviation against the 9-14h CDX change. A negative relation (prints tight of quotes when CDX tightens) is the composite lagging the market; that lag is observable at anchor time and enters Section 2 as `MKT_DEV_EFF_1H/3H`.

In [ ]:
# ---- bond-level persistence ---------------------------------------------------------------------------------------------
m = df.PREV_DEV_EFF.notna() & df.HOURS_SINCE_PREV_TRADE.gt(1 / 120)
p = df.loc[m, ["DEV_EFF", "PREV_DEV_EFF", "HOURS_SINCE_PREV_TRADE"]].copy()
p["a"] = bps(p.DEV_EFF).clip(-10, 10); p["b"] = bps(p.PREV_DEV_EFF).clip(-10, 10)
p["gap"] = pd.cut(p.HOURS_SINCE_PREV_TRADE, [0, 1 / 6, 1, 4, 24, 72, 1e9], labels=["<10min", "10-60min", "1-4h", "4-24h", "1-3d", ">3d"])
rows = [{"gap": "all", "n": len(p), "corr": p.a.corr(p.b), "slope": ols_slope(p.b, p.a)[0]}]
for lvl, g in p.groupby("gap", observed=True):
    rows.append({"gap": str(lvl), "n": len(g), "corr": g.a.corr(g.b), "slope": ols_slope(g.b, g.a)[0]})
pers = pd.DataFrame(rows)
print("persistence of the side-adjusted print-vs-CPP deviation between consecutive same-CUSIP prints (winsorised +-10 bps):")
print(pers.round(3).to_string(index=False))

# ---- market-level: hourly median deviation vs CDX ------------------------------------------------------------------------
_h = df.EFFECTIVE_DATETIME_TS.dt.hour
hourly = df.groupby(["DATE", _h]).agg(dev_med=("DEV_EFF", "median"), dev_mean=("DEV_EFF", lambda s: s.clip(-0.1, 0.1).mean()), n=("DEV_EFF", "size"))
hourly.index.names = ["DATE", "hour"]
hourly = hourly.reset_index()
if HAS["CDX_TRADE"]:
    cdx_h = df.groupby(["DATE", _h])["CDX_TRADE"].median().rename("cdx").reset_index().rename(columns={"EFFECTIVE_DATETIME_TS": "hour"})
    hourly = hourly.merge(cdx_h, on=["DATE", "hour"], how="left")
    daily = df.groupby("DATE").apply(lambda g: pd.Series({
        "dev_9_14": g.loc[g.EFFECTIVE_DATETIME_TS.dt.hour.between(9, 13), "DEV_EFF"].clip(-0.1, 0.1).mean() * 100,
        "cdx_chg_9_14": g.loc[g.EFFECTIVE_DATETIME_TS.dt.hour.eq(13), "CDX_TRADE"].median() - g.loc[g.EFFECTIVE_DATETIME_TS.dt.hour.eq(9), "CDX_TRADE"].median(),
        "cdx_range": g.CDX_TRADE.max() - g.CDX_TRADE.min()}), include_groups=False)
    print(f"\nacross {daily.dev_9_14.notna().sum()} days: corr(mean 9-14h side-adjusted deviation, 9-14h CDX change) = {daily.dev_9_14.corr(daily.cdx_chg_9_14):+.3f}")
    show_days = daily.cdx_range.sort_values(ascending=False).head(PLOT_DAYS).index
else:
    daily = None; show_days = hourly.groupby("DATE").n.sum().sort_values(ascending=False).head(PLOT_DAYS).index

fig, axes = plt.subplots(2, 3, figsize=(17, 8)); axes = axes.ravel()
for ax, dte in zip(axes, show_days):
    h = hourly[hourly.DATE.eq(dte)]
    ax.bar(h.hour, bps(h.dev_med), color=np.where(h.dev_med < 0, "tab:green", "tab:red"), alpha=0.7)
    ax.axhline(0, color="k", lw=0.8); ax.set_title(f"{dte:%Y-%m-%d}"); ax.set_ylabel("median print - side quote (bps)"); ax.set_xlabel("hour")
    if HAS["CDX_TRADE"] and h.cdx.notna().any():
        ax2 = ax.twinx(); ax2.plot(h.hour, h.cdx, color="tab:blue", marker="o", ms=3); ax2.set_ylabel("CDX level", color="tab:blue")
plt.suptitle("market-wide side-adjusted deviation by hour (bars) vs CDX (line): CPP lags the market on big-move days", y=1.02)
savefig("eda_1_4_market_cpp_lag_days")

if daily is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.regplot(data=daily, x="cdx_chg_9_14", y="dev_9_14", ax=axes[0], scatter_kws={"s": 18, "alpha": 0.7}, line_kws={"color": "k", "lw": 1})
    axes[0].set_xlabel("CDX change 9h -> 13h (bps)"); axes[0].set_ylabel("mean print - side quote, 9-14h (bps)"); axes[0].set_title("daily: composite lag vs market direction")
    axes[1].bar(pers.gap.astype(str), pers["corr"], color="tab:purple", alpha=0.8); axes[1].set_title("bond-level persistence of side-adjusted deviation\ncorr(dev_k, dev_k-1) by gap"); axes[1].set_ylabel("corr")
    savefig("eda_1_4_persistence")

### 1.5 Guardrail: is `TQW_SPREAD` at exec time a legitimate anchor-time quote?

The pipeline carries two Tradeweb columns: `TQW_SPREAD` (stamped at the print) and `MOST_RECENT_TQW_SPREAD` (the value as of the *previous* print, which the companion notebook uses for `TQW_DEV`). A post-trade evaluated price that already incorporates the print would look like a spectacular fair value and leak. Test: between consecutive same-CUSIP prints, correlate the change in `TQW_SPREAD` with the *current* print's deviation from the previous `TQW_SPREAD` and with the *previous* print's deviation. A legitimate quote follows the previous print (like CPP does); a contaminated one follows the current print. The cell prints both correlations and the share of exact matches with the print.

In [ ]:
if HAS["TQW_SPREAD"]:
    _gc = df.groupby("CUSIP", sort=False)
    df["P_TQW"] = _gc["TQW_SPREAD"].shift(1)
    m = df.P_TQW.notna() & df.SAME_DAY_PREV & df.HOURS_SINCE_PREV_TRADE.gt(1 / 120)
    t = df.loc[m, ["TQW_SPREAD", "P_TQW", "BM_SPREAD", "P_BM_SPREAD", "MID_SPREAD_CPP", "P_MID_SPREAD_CPP"]].astype(float)

    def quote_follow(q_now, q_prev):
        #   regress quote change on BOTH the current print's and the previous print's deviation from the previous quote:
        #   a legitimate quote loads on the previous print (it has been observed), a contaminated one on the current print.
        dq = (q_now - q_prev).to_numpy(); cur = (t.BM_SPREAD - q_prev).to_numpy(); prv = (t.P_BM_SPREAD - q_prev).to_numpy()
        ok = np.isfinite(dq) & np.isfinite(cur) & np.isfinite(prv) & (np.abs(cur) < 0.3) & (np.abs(prv) < 0.3)
        A = np.column_stack([np.ones(ok.sum()), cur[ok], prv[ok]])
        b = np.linalg.lstsq(A, dq[ok], rcond=None)[0]
        return b[1], b[2], np.corrcoef(dq[ok], cur[ok])[0, 1], np.corrcoef(dq[ok], prv[ok])[0, 1]
    r_t = quote_follow(t.TQW_SPREAD, t.P_TQW); r_c = quote_follow(t.MID_SPREAD_CPP, t.P_MID_SPREAD_CPP)
    res = pd.DataFrame({"TQW_SPREAD": [*r_t, (df.BM_SPREAD - df.TQW_SPREAD).abs().lt(5e-5).mean()],
                        "CPP mid": [*r_c, (df.BM_SPREAD - df.MID_SPREAD_CPP).abs().lt(5e-5).mean()]},
                       index=["coef on CURRENT print deviation  [leak if large]", "coef on PREVIOUS print deviation [legit]",
                              "corr(quote change, CURRENT print dev)", "corr(quote change, PREVIOUS print dev)", "share of prints equal to the quote to 0.005 bps"])
    print(res.round(3).to_string())
    leaky = (r_t[0] > 0.4) or (r_t[0] > r_c[0] + 0.2) or (res.iloc[4, 0] > 5 * max(res.iloc[4, 1], 1e-4))
    print(f"\nverdict: TQW_SPREAD at exec time is {'CONTAMINATED by the current print -> must not be used as an anchor-time feature' if leaky else 'not obviously contaminated (check manually before use)'}; "
          f"MOST_RECENT_TQW_SPREAD (previous-print value) is the safe version and is what TQW_DEV uses.")
    df = df.drop(columns=["P_TQW"])
else:
    print("TQW_SPREAD not in the tape; skipped.")

## 2. New anchor-time features

Every feature below uses only prints strictly before the anchor (or, for same-second flags, prints that are simultaneous with it), so it is available when the anchor is observed. Blocks:

| block | features | economic question |
| --- | --- | --- |
| **SIDE** | `EFF_SIDE`, `IS_TRUE_DD`, `IS_MISLABELED_D`, `IS_MARKUP`, `IS_ATS`, `IS_RP_PAIR`, `IS_RP_D_LEG`, `N_CUSIP_SEC`, `N_PRINTS_SAME_SEC_CUSIP`, `BURST_PURITY`, `IS_BURST`, `IS_SESSION`, `IS_PT_BURST`, `IS_EOD`, `N_PRINTS_TRAIL_60S`, `IS_144A_FLAG` | how much of the anchor print's deviation from CPP is information vs execution noise? |
| **CPPBIAS** | `PREV_DEV_EFF`, `ROLL_DEV_EFF_5P/20P`, `CUSIP_DEV_EFF_TODAY`, `CUSIP_DEV_EFF_PREV_D/5D`, `ISS_DEV_EFF_TODAY/PREV_D`, `MKT_DEV_EFF_1H/3H/PREV_D`, `CPP_MID_CHG_SINCE_PREV/24H`, `CPP_WIDTH_CHG_SINCE_PREV`, `CPP_WIDTH_REL`, `MKT_CPP_CHG_1H`, `SESSION_DEV_MED`, `SESSION_DEV_MED_TENOR`, `HOURS_SINCE_SESSION` | is CPP biased for this bond / issuer / market right now, and is it moving? |
| **MULTIDAY** | `FLOW_5D/20D`, `FLOW_TODAY_EXCL`, `ISS_FLOW_5D/20D`, `LAST_BLOCK_HOURS`, `LAST_BLOCK_SIGN`, `RV_5D/20D` (I-spread vs own history), `RV_5D_NET` (net of issuer), `ISS_CURVE_RESID`, `N_PRINTS_TODAY_SO_FAR`, `N_PRINTS_5D`, `TRADE_COUNTS_PREV_MONTH`, `IF_LIQUID_BOND`, `DAYS_SINCE_FIRST_PRINT`, `IS_NEW_ISSUE_20D` | dealer inventory pressure and relative-value drift on a multi-day horizon; liquidity; new-issue dynamics |
| **TIME** | `MINUTES_TO_CLOSE`, `IS_MORNING_ANCHOR`, `TARGET_DOW`, `IS_TARGET_MONTH_END_WEEK`, `IS_OVER_WEEKEND` | exposure of the target window to the open and to calendar flows |

Sign conventions follow the companion notebook: spread units (0.01 = 1 bp); signed flow is $+$ for B (dealer buys from customer) and $-$ for S.

### 2.1 CPP-bias / CPP-lag block

In [ ]:
df = df.sort_values(["CUSIP", "EFFECTIVE_DATETIME_TS", "TRADE_ROW_ID"], kind="stable").reset_index(drop=True)
_gc = df.groupby("CUSIP", sort=False)
_dev = df["DEV_EFF"].clip(-0.15, 0.15)   # clipped at 15 bps for all rolling means
df["_DEV_C"] = _dev

# per-CUSIP: previous prints (strictly before)
df["ROLL_DEV_EFF_5P"] = _gc["_DEV_C"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean()).astype("float32")
df["ROLL_DEV_EFF_20P"] = _gc["_DEV_C"].transform(lambda s: s.shift(1).rolling(20, min_periods=3).mean()).astype("float32")
_gcd = df.groupby(["CUSIP", "DATE"], sort=False)
_cs = _gcd["_DEV_C"].cumsum() - df["_DEV_C"]; _cn = _gcd.cumcount()
df["CUSIP_DEV_EFF_TODAY"] = (_cs / _cn.where(_cn > 0)).astype("float32")
df["N_PRINTS_TODAY_SO_FAR"] = _cn.astype("int16")

# trading-day grid helpers -------------------------------------------------------------------------------------------------
trading_days = np.array(sorted(df["DATE"].unique()), dtype="datetime64[ns]")
day_pos = pd.Series(np.arange(len(trading_days)), index=pd.DatetimeIndex(trading_days))
df["DAY_IDX"] = day_pos.reindex(df["DATE"].to_numpy()).to_numpy().astype("int32")

def grid_rolling(daily, key, val_cols, windows, agg="sum", min_periods=1):
    #   daily: frame with [key, DAY_IDX, val_cols]; returns frame [key, DAY_IDX, f"{v}_W{w}"] with the rolling stat over the
    #   PREVIOUS w trading days (today excluded), computed on a dense key x day grid so calendar gaps are handled.
    keys = daily[key].unique(); kpos = pd.Series(np.arange(len(keys)), index=keys)
    n_k, n_d = len(keys), len(trading_days)
    out = {}
    for v in val_cols:
        grid = np.full((n_k, n_d), np.nan)
        grid[kpos.reindex(daily[key]).to_numpy(), daily["DAY_IDX"].to_numpy()] = daily[v].to_numpy(dtype="float64")
        g = pd.DataFrame(grid.T)   # days x keys
        for w in windows:
            r = getattr(g.rolling(w, min_periods=min_periods), agg)().shift(1)   # previous w days, today excluded
            out[f"{v}_W{w}"] = r.to_numpy().T.ravel()
    idx = pd.MultiIndex.from_product([keys, np.arange(n_d)], names=[key, "DAY_IDX"])
    return pd.DataFrame(out, index=idx).reset_index()

# per-CUSIP daily mean deviation -> previous day and previous 5 days
cd = df.groupby(["CUSIP", "DAY_IDX"], sort=False).agg(dev_sum=("_DEV_C", "sum"), n=("_DEV_C", "size")).reset_index()
cr = grid_rolling(cd, "CUSIP", ["dev_sum", "n"], [1, 5])
for w in [1, 5]:
    cr[f"CUSIP_DEV_EFF_W{w}"] = cr[f"dev_sum_W{w}"] / cr[f"n_W{w}"].where(cr[f"n_W{w}"] > 0)
df = df.merge(cr[["CUSIP", "DAY_IDX", "CUSIP_DEV_EFF_W1", "CUSIP_DEV_EFF_W5"]], on=["CUSIP", "DAY_IDX"], how="left")
df = df.rename(columns={"CUSIP_DEV_EFF_W1": "CUSIP_DEV_EFF_PREV_D", "CUSIP_DEV_EFF_W5": "CUSIP_DEV_EFF_PREV_5D"})

# issuer level: previous day and today-so-far (excluding the current print), time-ordered
df = df.sort_values(["EFFECTIVE_DATETIME_TS", "TRADE_ROW_ID"], kind="stable").reset_index(drop=True)
_gid = df.groupby(["ISSUER", "DATE"], sort=False)
_s = _gid["_DEV_C"].cumsum() - df["_DEV_C"]; _n = _gid.cumcount()
df["ISS_DEV_EFF_TODAY"] = (_s / _n.where(_n > 0)).astype("float32")
idd = df.groupby(["ISSUER", "DAY_IDX"], sort=False).agg(dev_sum=("_DEV_C", "sum"), n=("_DEV_C", "size")).reset_index()
ir = grid_rolling(idd, "ISSUER", ["dev_sum", "n"], [1])
ir["ISS_DEV_EFF_PREV_D"] = ir["dev_sum_W1"] / ir["n_W1"].where(ir["n_W1"] > 0)
df = df.merge(ir[["ISSUER", "DAY_IDX", "ISS_DEV_EFF_PREV_D"]], on=["ISSUER", "DAY_IDX"], how="left")

# market level: trailing windows strictly before t (clipped mean), and previous day
def trailing_mean(values, hours):
    t = df["EFFECTIVE_DATETIME_TS"].astype("int64").to_numpy()
    v = np.nan_to_num(np.asarray(values, dtype="float64"), nan=0.0); ok = np.isfinite(np.asarray(values, dtype="float64")).astype("float64")
    cs, cn = np.concatenate([[0.0], np.cumsum(v)]), np.concatenate([[0.0], np.cumsum(ok)])
    hi = np.searchsorted(t, t, side="left"); lo = np.searchsorted(t, t - int(hours * 3600 * 1e9), side="left")
    n = cn[hi] - cn[lo]
    return np.where(n >= 20, (cs[hi] - cs[lo]) / np.where(n > 0, n, 1), np.nan).astype("float32")
df["MKT_DEV_EFF_1H"] = trailing_mean(df["_DEV_C"], 1.0)
df["MKT_DEV_EFF_3H"] = trailing_mean(df["_DEV_C"], 3.0)
md_ = df.groupby("DAY_IDX", sort=False)["_DEV_C"].mean().rename("MKT_DEV_EFF_PREV_D")
df["MKT_DEV_EFF_PREV_D"] = md_.shift(1).reindex(df["DAY_IDX"].to_numpy()).to_numpy().astype("float32")
df["MKT_CPP_CHG_1H"] = trailing_mean(df["CPP_MID_CHG_SINCE_PREV"].where(df["HOURS_SINCE_PREV_TRADE"].lt(6)).clip(-0.15, 0.15), 1.0)

# CPP momentum over 24h (same CUSIP print >= 24h ago), roll-corrected
_sub = df[["CUSIP", "EFFECTIVE_DATETIME_TS", "MID_SPREAD_CPP", "BM_CUSIP"]].copy()
_sub["MERGE_TS"] = _sub["EFFECTIVE_DATETIME_TS"] + pd.Timedelta(hours=24)
_sub = _sub.rename(columns={"MID_SPREAD_CPP": "MID_CPP_24H", "BM_CUSIP": "BM_CUSIP_24H", "EFFECTIVE_DATETIME_TS": "TS_24H"}).sort_values("MERGE_TS")
df = pd.merge_asof(df.sort_values("EFFECTIVE_DATETIME_TS"), _sub, by="CUSIP", left_on="EFFECTIVE_DATETIME_TS", right_on="MERGE_TS", allow_exact_matches=False, direction="backward").drop(columns=["MERGE_TS"])
_off24 = bm_roll_offset(df["BM_CUSIP"].to_numpy(), df["BM_CUSIP_24H"].to_numpy(), df["DATE"].to_numpy())
df["CPP_MID_CHG_24H"] = (df["MID_SPREAD_CPP"] - df["MID_CPP_24H"] + _off24).astype("float32")
df = df.drop(columns=["MID_CPP_24H", "BM_CUSIP_24H", "TS_24H"])
df = df.sort_values(["CUSIP", "EFFECTIVE_DATETIME_TS", "TRADE_ROW_ID"], kind="stable").reset_index(drop=True)
df["CPP_WIDTH_REL"] = (df["CPP_WIDTH"] / df.groupby("CUSIP", sort=False)["CPP_WIDTH"].transform(lambda s: s.shift(1).rolling(20, min_periods=3).median())).astype("float32")

# interdealer session snapshots: median (print - CPP mid) per session, overall and by BM tenor; attached strictly after the session
sess = df[df.IS_SESSION.eq(1)]
if len(sess):
    s_all = sess.groupby("SEC").agg(SESSION_DEV_MED=("DEV_MID", "median"), SESSION_N=("DEV_MID", "size")).reset_index().sort_values("SEC")
    s_ten = sess.groupby(["SEC", "BM_TENOR_GROUP"]).agg(SESSION_DEV_MED_TENOR=("DEV_MID", "median"), n=("DEV_MID", "size")).reset_index()
    s_ten = s_ten[s_ten.n.ge(5)].drop(columns="n").sort_values("SEC")
    df = df.sort_values("EFFECTIVE_DATETIME_TS")
    df = pd.merge_asof(df, s_all.rename(columns={"SEC": "SESSION_SEC"}), left_on="EFFECTIVE_DATETIME_TS", right_on="SESSION_SEC", allow_exact_matches=False, direction="backward")
    df["BM_TENOR_GROUP"] = df["BM_TENOR_GROUP"].astype("int64", errors="ignore")
    s_ten["BM_TENOR_GROUP"] = s_ten["BM_TENOR_GROUP"].astype(df["BM_TENOR_GROUP"].dtype)
    df = pd.merge_asof(df, s_ten.rename(columns={"SEC": "SESSION_SEC_T"}), by="BM_TENOR_GROUP", left_on="EFFECTIVE_DATETIME_TS", right_on="SESSION_SEC_T", allow_exact_matches=False, direction="backward")
    df["HOURS_SINCE_SESSION"] = ((df["EFFECTIVE_DATETIME_TS"] - df["SESSION_SEC"]).dt.total_seconds() / 3600).astype("float32")
    df = df.drop(columns=["SESSION_SEC", "SESSION_SEC_T", "SESSION_N"])
    print(f"{len(s_all):,} interdealer sessions detected ({sess.DATE.nunique()} days, median size {s_all.SESSION_N.median():.0f} prints); "
          f"session median print-CPP mid: mean {100 * s_all.SESSION_DEV_MED.mean():+.2f} bps, std {100 * s_all.SESSION_DEV_MED.std():.2f} bps")
    fig, ax = plt.subplots(figsize=(14, 3.8))
    ax.scatter(s_all.SEC, bps(s_all.SESSION_DEV_MED), s=np.clip(s_all.SESSION_N / 5, 5, 60), alpha=0.6); ax.axhline(0, color="k", lw=0.8)
    ax.set_title("interdealer sessions: median print - CPP mid (bps); size = session size"); ax.set_ylabel("bps")
    savefig("feat_2_1_sessions")
else:
    for c in ["SESSION_DEV_MED", "SESSION_DEV_MED_TENOR", "HOURS_SINCE_SESSION"]:
        df[c] = np.float32(np.nan)
    print("no interdealer sessions detected at the current threshold; session features are NaN")
for c in ["SESSION_DEV_MED", "SESSION_DEV_MED_TENOR"]:
    df[c] = df[c].where(df["HOURS_SINCE_SESSION"].lt(8)).astype("float32")   # stale sessions carry no information
df = df.drop(columns=["_DEV_C"])

CPPBIAS_FEATURES = ["PREV_DEV_EFF", "ROLL_DEV_EFF_5P", "ROLL_DEV_EFF_20P", "CUSIP_DEV_EFF_TODAY", "CUSIP_DEV_EFF_PREV_D", "CUSIP_DEV_EFF_PREV_5D",
                    "ISS_DEV_EFF_TODAY", "ISS_DEV_EFF_PREV_D", "MKT_DEV_EFF_1H", "MKT_DEV_EFF_3H", "MKT_DEV_EFF_PREV_D",
                    "CPP_MID_CHG_SINCE_PREV", "CPP_MID_CHG_24H", "CPP_WIDTH_CHG_SINCE_PREV", "CPP_WIDTH_REL", "MKT_CPP_CHG_1H",
                    "SESSION_DEV_MED", "SESSION_DEV_MED_TENOR", "HOURS_SINCE_SESSION"]
print(bps(df[CPPBIAS_FEATURES[:11]]).describe().T[["count", "mean", "std", "25%", "50%", "75%"]].round(3).to_string())

### 2.2 Multi-day flow, relative value, liquidity and new-issue block

- **Flow**: signed customer volume ($+$ dealer buys, $-$ dealer sells, 0 interdealer) over the previous 5 / 20 trading days divided by gross volume, at bond and issuer level; today's imbalance so far excluding the current print; hours since the last $\ge$5MM block in the bond and its sign. Rationale: a dealer who bought a block is long inventory and offers it out over the following days.
- **Relative value on I-spread**: `I_SPREAD` (spread to swaps) has no benchmark-roll problem, so multi-day level changes are clean. `RV_5D` = current I-spread minus the median of the bond's daily median I-spread over the previous 5 trading days; `RV_20D` likewise; `RV_5D_NET` subtracts the issuer's previous-day median of the same quantity across its bonds. `ISS_CURVE_RESID`: residual of the anchor's I-spread against the issuer's own curve $y = a + b\log(\text{years})$ fitted on the previous 5 days' prints (needs $\ge$10 prints and dispersion in maturity). Falls back to `BM_SPREAD` with a warning if `I_SPREAD` is absent.
- **Liquidity**: `TRADE_COUNTS_PREV_MONTH`, `IF_LIQUID_BOND` (both pipeline columns, unused so far), `N_PRINTS_TODAY_SO_FAR`, `N_PRINTS_5D`.
- **New issue**: `DAYS_SINCE_FIRST_PRINT` (trading days since the CUSIP's first print in the tape; censored when the first print is within the first 20 tape days), `IS_NEW_ISSUE_20D`.

In [ ]:
df = df.sort_values(["EFFECTIVE_DATETIME_TS", "TRADE_ROW_ID"], kind="stable").reset_index(drop=True)
_sgn = df["EFF_SIDE"].map({"B": 1.0, "S": -1.0}).fillna(0.0).to_numpy()
df["_SIGNED_Q"] = _sgn * df["QUANTITY"].to_numpy(); df["_GROSS_Q"] = df["QUANTITY"].to_numpy()

# today so far (excluding the current print)
for key, name in [(["CUSIP", "DATE"], "FLOW_TODAY_EXCL"), (["ISSUER", "DATE"], "ISS_FLOW_TODAY_EXCL")]:
    g = df.groupby(key, sort=False)
    s = g["_SIGNED_Q"].cumsum() - df["_SIGNED_Q"]; q = g["_GROSS_Q"].cumsum() - df["_GROSS_Q"]
    df[name] = (s / q.where(q > 0)).astype("float32")

# previous 5 / 20 trading days on the dense grid
cd = df.groupby(["CUSIP", "DAY_IDX"], sort=False).agg(sq=("_SIGNED_Q", "sum"), gq=("_GROSS_Q", "sum"), n=("_GROSS_Q", "size")).reset_index()
cr = grid_rolling(cd, "CUSIP", ["sq", "gq", "n"], [5, 20])
for w in [5, 20]:
    cr[f"FLOW_{w}D"] = cr[f"sq_W{w}"] / cr[f"gq_W{w}"].where(cr[f"gq_W{w}"] > 0)
cr["N_PRINTS_5D"] = cr["n_W5"].fillna(0)
df = df.merge(cr[["CUSIP", "DAY_IDX", "FLOW_5D", "FLOW_20D", "N_PRINTS_5D"]], on=["CUSIP", "DAY_IDX"], how="left")
idd = df.groupby(["ISSUER", "DAY_IDX"], sort=False).agg(sq=("_SIGNED_Q", "sum"), gq=("_GROSS_Q", "sum")).reset_index()
ir = grid_rolling(idd, "ISSUER", ["sq", "gq"], [5, 20])
for w in [5, 20]:
    ir[f"ISS_FLOW_{w}D"] = ir[f"sq_W{w}"] / ir[f"gq_W{w}"].where(ir[f"gq_W{w}"] > 0)
df = df.merge(ir[["ISSUER", "DAY_IDX", "ISS_FLOW_5D", "ISS_FLOW_20D"]], on=["ISSUER", "DAY_IDX"], how="left")

# last >= 5MM block in the bond, strictly before t
blk = df.loc[df.QUANTITY.ge(5e6) & df.EFF_SIDE.isin(["B", "S"]), ["CUSIP", "EFFECTIVE_DATETIME_TS", "EFF_SIDE"]].copy()
blk["LAST_BLOCK_SIGN"] = blk["EFF_SIDE"].map({"B": 1.0, "S": -1.0}).astype("float32")
blk = blk.rename(columns={"EFFECTIVE_DATETIME_TS": "BLOCK_TS"}).drop(columns="EFF_SIDE").sort_values("BLOCK_TS")
df = pd.merge_asof(df.sort_values("EFFECTIVE_DATETIME_TS"), blk, by="CUSIP", left_on="EFFECTIVE_DATETIME_TS", right_on="BLOCK_TS", allow_exact_matches=False, direction="backward")
df["LAST_BLOCK_HOURS"] = ((df["EFFECTIVE_DATETIME_TS"] - df["BLOCK_TS"]).dt.total_seconds() / 3600).astype("float32")
df["LAST_BLOCK_SIGN"] = df["LAST_BLOCK_SIGN"].where(df["LAST_BLOCK_HOURS"].lt(24 * 7)).fillna(0.0).astype("float32")
df = df.drop(columns=["BLOCK_TS"])

# relative value on I-spread (spread units)
if HAS["I_SPREAD"]:
    df["ISPRD"] = pd.to_numeric(df["I_SPREAD"], errors="coerce") / 100.0
else:
    print("WARNING: I_SPREAD missing -> RV features use BM_SPREAD (benchmark rolls not corrected in multi-day medians)")
    df["ISPRD"] = df["BM_SPREAD"]
_ok = df["ISPRD"].between(-1.0, 10.0)
df.loc[~_ok, "ISPRD"] = np.nan
cd = df.groupby(["CUSIP", "DAY_IDX"], sort=False).agg(med=("ISPRD", "median")).reset_index()
cr = grid_rolling(cd, "CUSIP", ["med"], [5, 20], agg="median", min_periods=1)
df = df.merge(cr.rename(columns={"med_W5": "ISPRD_MED_5D", "med_W20": "ISPRD_MED_20D"}), on=["CUSIP", "DAY_IDX"], how="left")
df["RV_5D"] = (df["ISPRD"] - df["ISPRD_MED_5D"]).astype("float32")
df["RV_20D"] = (df["ISPRD"] - df["ISPRD_MED_20D"]).astype("float32")
# issuer-level previous-day median of the bond-level 5d move
cd2 = cd.merge(cr[["CUSIP", "DAY_IDX", "med_W5"]], on=["CUSIP", "DAY_IDX"], how="left")
cd2["chg"] = cd2["med"] - cd2["med_W5"]
cd2 = cd2.merge(df[["CUSIP", "ISSUER"]].drop_duplicates("CUSIP"), on="CUSIP", how="left")
idd = cd2.groupby(["ISSUER", "DAY_IDX"]).agg(chg_med=("chg", "median")).reset_index()
ir = grid_rolling(idd, "ISSUER", ["chg_med"], [1], agg="mean")
df = df.merge(ir.rename(columns={"chg_med_W1": "ISS_RV_5D_PREV_D"}), on=["ISSUER", "DAY_IDX"], how="left")
df["RV_5D_NET"] = (df["RV_5D"] - df["ISS_RV_5D_PREV_D"].fillna(0.0)).astype("float32")

# issuer curve residual: OLS of ISPRD on log(years) over the previous 5 days' prints, closed form from rolling sums
_x = np.log(df["YRS_TO_MATURITY"].clip(lower=0.1)); _y = df["ISPRD"]
_ok = _x.notna() & _y.notna()
cs = pd.DataFrame({"ISSUER": df.loc[_ok, "ISSUER"], "DAY_IDX": df.loc[_ok, "DAY_IDX"], "n": 1.0, "sx": _x[_ok], "sy": _y[_ok], "sxx": _x[_ok] ** 2, "sxy": _x[_ok] * _y[_ok]})
cs = cs.groupby(["ISSUER", "DAY_IDX"]).sum().reset_index()
cr = grid_rolling(cs, "ISSUER", ["n", "sx", "sy", "sxx", "sxy"], [5], agg="sum")
n_, sx, sy, sxx, sxy = (cr[f"{c}_W5"] for c in ["n", "sx", "sy", "sxx", "sxy"])
varx = sxx / n_ - (sx / n_) ** 2
b = (sxy - sx * sy / n_) / (varx * n_)
a = (sy - b * sx) / n_
good = n_.ge(10) & varx.ge(0.02)
cr["CURVE_A"] = a.where(good); cr["CURVE_B"] = b.where(good)
df = df.merge(cr[["ISSUER", "DAY_IDX", "CURVE_A", "CURVE_B"]], on=["ISSUER", "DAY_IDX"], how="left")
df["ISS_CURVE_RESID"] = (df["ISPRD"] - (df["CURVE_A"] + df["CURVE_B"] * _x)).astype("float32")
df = df.drop(columns=["CURVE_A", "CURVE_B", "ISPRD_MED_5D", "ISPRD_MED_20D", "ISS_RV_5D_PREV_D", "_SIGNED_Q", "_GROSS_Q"])

# liquidity and new-issue age
df["TRADE_COUNTS_PREV_MONTH"] = pd.to_numeric(df["TRADE_COUNTS_PREV_MONTH"], errors="coerce").astype("float32") if HAS["TRADE_COUNTS_PREV_MONTH"] else np.float32(np.nan)
df["IF_LIQUID_BOND"] = df["IF_LIQUID_BOND"].astype("float32") if HAS["IF_LIQUID_BOND"] else np.float32(np.nan)
first_day = df.groupby("CUSIP", sort=False)["DAY_IDX"].transform("min")
df["DAYS_SINCE_FIRST_PRINT"] = (df["DAY_IDX"] - first_day).astype("float32")
_censored = first_day.lt(20)
df.loc[_censored, "DAYS_SINCE_FIRST_PRINT"] = np.nan
df["IS_NEW_ISSUE_20D"] = (df["DAYS_SINCE_FIRST_PRINT"].lt(20)).astype("int8")

MULTIDAY_FEATURES = ["FLOW_5D", "FLOW_20D", "FLOW_TODAY_EXCL", "ISS_FLOW_5D", "ISS_FLOW_20D", "ISS_FLOW_TODAY_EXCL", "LAST_BLOCK_HOURS", "LAST_BLOCK_SIGN",
                     "RV_5D", "RV_20D", "RV_5D_NET", "ISS_CURVE_RESID", "N_PRINTS_TODAY_SO_FAR", "N_PRINTS_5D", "TRADE_COUNTS_PREV_MONTH", "IF_LIQUID_BOND",
                     "DAYS_SINCE_FIRST_PRINT", "IS_NEW_ISSUE_20D"]
print(f"new issues (first print within the tape, >20 tape-days after its start): {df.loc[df.DAYS_SINCE_FIRST_PRINT.notna(), 'CUSIP'].nunique():,} CUSIPs; "
      f"prints within 20 days of first print: {df.IS_NEW_ISSUE_20D.mean():.1%}")
print(df[MULTIDAY_FEATURES].describe().T[["count", "mean", "std", "25%", "50%", "75%"]].round(3).to_string())

### 2.3 Time and calendar block, the compact "old core", and side features

**TIME**: `MINUTES_TO_CLOSE`, `IS_MORNING_ANCHOR` (anchor before 11:30, so the $\pm$2h target window contains the open), and two target-day regimes with $\le$5 levels each: `TARGET_DOW` and `IS_TARGET_MONTH_END_WEEK` (target date within the last three trading days of its month). Regime features are only allowed to interact with bond-level features in LightGBM, as in the companion notebook.

**OLD_CORE**: a compact reproduction (about 25 features) of the companion notebook's signals that carried permutation value, so that the new blocks are tested *incrementally* against a fair reference rather than in isolation. It is not the full 82-feature stack; the point is the delta from adding blocks.

In [ ]:
df = df.sort_values(["EFFECTIVE_DATETIME_TS", "TRADE_ROW_ID"], kind="stable").reset_index(drop=True)
df["IS_MORNING_ANCHOR"] = df["HOUR_F"].lt(11.5).astype("int8")

# trading-day calendar and target-day regimes (T+1)
market_schedule = pd.DataFrame({"TRADE_DATE": pd.DatetimeIndex(trading_days).tz_localize("America/New_York")})
market_schedule["TRADE_DATE_PLUS_1"] = market_schedule["TRADE_DATE"].shift(-1)
_t1 = pd.Series(market_schedule["TRADE_DATE_PLUS_1"].dt.tz_localize(None).to_numpy(), index=pd.DatetimeIndex(trading_days))
_t1_of_row = _t1.reindex(df["DATE"].to_numpy())
df["TARGET_DOW"] = pd.DatetimeIndex(_t1_of_row.to_numpy()).dayofweek.astype("float32")
_month = pd.Series(pd.DatetimeIndex(trading_days).to_period("M"), index=pd.DatetimeIndex(trading_days))
_rank_from_end = _month.groupby(_month).cumcount(ascending=False)
_me = pd.Series(_rank_from_end.to_numpy() < 3, index=pd.DatetimeIndex(trading_days))
df["IS_TARGET_MONTH_END_WEEK"] = _me.reindex(_t1_of_row.to_numpy()).fillna(False).to_numpy().astype("float32")
TIME_FEATURES = ["MINUTES_TO_CLOSE", "IS_MORNING_ANCHOR", "TARGET_DOW", "IS_TARGET_MONTH_END_WEEK"]   # IS_OVER_WEEKEND is pair-level (below)

# ---- compact old core (companion-notebook signals) ------------------------------------------------------------------------
df["CPP_MID_DEV"] = df["DEV_MID"].astype("float32")
df["CPP_MID_DEV_NORM"] = (df["DEV_MID"] / df["CPP_WIDTH"]).astype("float32")
xs = df.groupby("DATE")["BM_SPREAD_DEV"].agg(["mean", "std"]).shift(1) if HAS["BM_SPREAD_DEV"] else None
if xs is not None:
    df = df.join(xs.rename(columns={"mean": "_m", "std": "_s"}), on="DATE")
    df["XS_SPREAD_DEV_Z"] = ((pd.to_numeric(df["BM_SPREAD_DEV"], errors="coerce") - df["_m"]) / df["_s"]).astype("float32")
    df = df.drop(columns=["_m", "_s"])
df["D_BM_SPREAD_PER_SQRT_HOUR"] = (pd.to_numeric(df["D_BM_SPREAD"], errors="coerce") / np.sqrt(df["HOURS_SINCE_PREV_TRADE"].clip(lower=0.25))).astype("float32") if HAS["D_BM_SPREAD"] else np.float32(np.nan)
if HAS["MOST_RECENT_TQW_SPREAD"]:
    df["TQW_DEV"] = (df["BM_SPREAD"] - pd.to_numeric(df["MOST_RECENT_TQW_SPREAD"], errors="coerce")).astype("float32")
    if HAS["MOST_RECENT_EXEC_TIME"]:
        df["TQW_STALE_MIN"] = ((df["EFFECTIVE_DATETIME_TS"].dt.tz_localize(None) - pd.to_datetime(df["MOST_RECENT_EXEC_TIME"], errors="coerce")).dt.total_seconds() / 60).astype("float32")
_sg = df["TRADE_TYPE"].map({"B": 1.0, "S": -1.0}).fillna(0.0) * df["QUANTITY"]
for c, k in [("FLOW_IMBALANCE_TODAY", ["CUSIP", "DATE"]), ("ISSUER_FLOW_IMBALANCE_TODAY", ["ISSUER", "DATE"])]:
    df[c] = (_sg.groupby([df[j] for j in k]).cumsum() / df["QUANTITY"].groupby([df[j] for j in k]).cumsum()).astype("float32")
df["LOG_QTY"] = np.log10(df["QUANTITY"].clip(lower=1)).astype("float32")
df["LOG_PREV_QTY"] = np.log10(pd.to_numeric(df["PREV_QUANTITY"], errors="coerce").clip(lower=1)).astype("float32") if HAS["PREV_QUANTITY"] else np.float32(np.nan)

OLD_CORE = [c for c in ["CPP_MID_DEV", "CPP_MID_DEV_NORM", "CPP_WIDTH", "D_BM_SPREAD", "BM_SPREAD_DEV", "XS_SPREAD_DEV_Z", "HOURS_SINCE_PREV_TRADE",
                        "D_BM_SPREAD_PER_SQRT_HOUR", "TQW_DEV", "TQW_STALE_MIN", "FLOW_IMBALANCE_TODAY", "ISSUER_FLOW_IMBALANCE_TODAY", "LOG_QTY", "LOG_PREV_QTY",
                        "YRS_TO_MATURITY", "COUPON", "IS_SIZE_ESTIMATED", "TRADE_TYPE", "PREV_TRADE_TYPE", "MEAN_ISSUER_SPREAD_DEV", "NUM_OF_ISSUER_TRADES_SINCE_PREV",
                        "BM_SPREAD_STD_GROUP_BY_TYPE", "BM_YIELD_STD", "ROLLING_BM_SPREAD", "BM_SPREAD", "D_CDX_TRADE", "CDX_TRADE_DEV"] if c in df.columns]
for c in ["D_BM_SPREAD", "BM_SPREAD_DEV", "COUPON", "MEAN_ISSUER_SPREAD_DEV", "NUM_OF_ISSUER_TRADES_SINCE_PREV", "BM_SPREAD_STD_GROUP_BY_TYPE", "BM_YIELD_STD", "ROLLING_BM_SPREAD", "D_CDX_TRADE", "CDX_TRADE_DEV"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")

SIDE_FEATURES = ["EFF_SIDE", "IS_TRUE_DD", "IS_MISLABELED_D", "IS_MARKUP", "IS_ATS", "IS_RP_PAIR", "IS_RP_D_LEG", "N_CUSIP_SEC", "N_PRINTS_SAME_SEC_CUSIP",
                 "BURST_PURITY", "IS_BURST", "IS_SESSION", "IS_PT_BURST", "IS_EOD", "N_PRINTS_TRAIL_60S", "IS_144A_FLAG"]
CAT_FEATURES = [c for c in ["TRADE_TYPE", "PREV_TRADE_TYPE", "TRADE_TYPE_NEXT", "EFF_SIDE", "EFF_SIDE_NEXT"]]
REGIME_LIKE = ["TARGET_DOW", "IS_TARGET_MONTH_END_WEEK", "MKT_DEV_EFF_PREV_D"]
print(f"OLD_CORE {len(OLD_CORE)} | SIDE {len(SIDE_FEATURES)} | CPPBIAS {len(CPPBIAS_FEATURES)} | MULTIDAY {len(MULTIDAY_FEATURES)} | TIME {len(TIME_FEATURES)}")

# keep only what the pairing and the evaluation need (every anchor column is replicated ~1.8x in the pair frame; raw object columns are the expensive ones)
KEEP_FOR_PAIRS = list(dict.fromkeys(
    ["TRADE_ROW_ID", "CUSIP", "ISSUER", "DATE", "DAY_IDX", "EFFECTIVE_DATETIME_TS", "BM_CUSIP", "BM_TENOR_GROUP", "BM_SPREAD", "BID_SPREAD_CPP", "ASK_SPREAD_CPP",
     "MID_SPREAD_CPP", "CPP_WIDTH", "TRADE_TYPE", "EFF_SIDE", "QUANTITY", "LOG_QTY", "IS_SIZE_ESTIMATED", "PRINT_CLASS", "HOUR_F", "HOURS_SINCE_PREV_TRADE",
     "DEV_EFF", "DEV_MID", *OLD_CORE, *SIDE_FEATURES, *CPPBIAS_FEATURES, *MULTIDAY_FEATURES, *TIME_FEATURES]))
df = df[[c for c in KEEP_FOR_PAIRS if c in df.columns]].copy()
for c in ["ISSUER", "PRINT_CLASS"]:
    df[c] = df[c].astype("category")
_KEEP_F64 = {"BM_SPREAD", "BID_SPREAD_CPP", "ASK_SPREAD_CPP", "MID_SPREAD_CPP", "QUANTITY", "YRS_TO_MATURITY", "CPP_WIDTH"}
df = df.astype({c: "float32" for c in df.columns if df[c].dtype == "float64" and c not in _KEEP_F64})
base_df = df.reset_index(drop=True)
print(f"base frame: {len(base_df):,} prints x {base_df.shape[1]} cols, {base_df.memory_usage(deep=False).sum() / 1e9:.2f} GB; elapsed {(time.time() - t_start) / 60:.1f} min")

## 3. Anchor -> T+1 target pairs, pair-level features, and the CPP-drift decomposition

The pairing is the companion notebook's `attach_forward_target_single_horizon` (same-CUSIP prints on the next trading day within $\pm$2h of the anchor clock time, benchmark-roll corrected). The target row's own quotes and flags are carried as `*_NEXT` columns so that, for each pair, we can build

- the **old** target-side quote deviation from `TRADE_TYPE_NEXT` (`CPP_TARGET_SIDE_DEV`, the companion's `cpp_side`) and the **new** one from `EFF_SIDE_NEXT` (`CPP_TARGET_SIDE_DEV_EFF`, `cpp_side_eff`), plus `SIDE_FLIP_EFF`, `EXPECTED_BOUNCE_EFF`;
- the scenario flags a trader knows when pricing the target (`EFF_SIDE_NEXT`, `IS_BURST_NEXT`, `IS_EOD_NEXT`);
- the exact decomposition $\text{TARGET} = \texttt{cpp\_side\_eff} + \texttt{CPP\_DRIFT} + \texttt{TARGET\_PRINT\_NOISE}$ with `CPP_DRIFT` $= q_{\ell_u}(u) - q_{\ell_u}(t) + r$ and the noise $= s_u - q_{\ell_u}(u)$, and a split of the mid-quote drift into rest-of-day-$T$ (anchor $\to$ last print of the bond on $T$), overnight (last print on $T$ $\to$ first print on $T{+}1$) and day-$T{+}1$ (first print $\to$ target).

In [ ]:
def attach_forward_target_single_horizon(df, bm_dict, trading_days=1):
    target_window_tolerance_hours = 2.0 * trading_days
    order = np.argsort(df["EFFECTIVE_DATETIME_TS"].to_numpy(), kind="stable")
    key_cols = [c for c in ["TRADE_ROW_ID", "CUSIP", "EFFECTIVE_DATETIME_TS", "DATE", "BM_CUSIP", "BM_SPREAD", "TRADE_TYPE", "QUANTITY",
                            "EFF_SIDE", "IS_BURST", "IS_EOD", "IS_SESSION", "BID_SPREAD_CPP", "ASK_SPREAD_CPP", "MID_SPREAD_CPP", "CPP_WIDTH"] if c in df.columns]
    out = df[key_cols].take(order); out.index = pd.RangeIndex(len(out))
    out["TRADE_DATE"] = out.EFFECTIVE_DATETIME_TS.dt.normalize()
    out = out.merge(market_schedule[["TRADE_DATE", f"TRADE_DATE_PLUS_{trading_days}"]], on="TRADE_DATE", how="left", sort=False)
    out["TARGET_ANCHOR_TIME"] = out["EFFECTIVE_DATETIME_TS"] + (out[f"TRADE_DATE_PLUS_{trading_days}"] - out["TRADE_DATE"])
    out["TARGET_WINDOW_START"] = out["TARGET_ANCHOR_TIME"] - pd.Timedelta(hours=target_window_tolerance_hours)
    out["TARGET_WINDOW_END"] = out["TARGET_ANCHOR_TIME"] + pd.Timedelta(hours=target_window_tolerance_hours)
    out["TARGET_ANCHOR_GAP_HOURS"] = (out["TARGET_ANCHOR_TIME"] - out["EFFECTIVE_DATETIME_TS"]).dt.total_seconds() / 3600
    out["_ANCHOR_ID"] = np.arange(len(out))
    target_cols = [c for c in key_cols if c in out.columns]
    targets = out[target_cols].sort_values(["CUSIP", "EFFECTIVE_DATETIME_TS", "TRADE_ROW_ID"]).reset_index(drop=True)
    s_arr = pd.to_datetime(out["TARGET_WINDOW_START"], errors="coerce").to_numpy(dtype="datetime64[ns]")
    e_arr = pd.to_datetime(out["TARGET_WINDOW_END"], errors="coerce").to_numpy(dtype="datetime64[ns]")
    t_arr = pd.to_datetime(targets["EFFECTIVE_DATETIME_TS"], errors="coerce").to_numpy(dtype="datetime64[ns]")
    tgi = targets.groupby("CUSIP", sort=False).indices; agi = out.groupby("CUSIP", sort=False).indices
    a_chunks, t_chunks = [], []
    for cusip, a_idx in agi.items():
        t_idx = tgi.get(cusip)
        if t_idx is None or len(t_idx) == 0: continue
        tt = t_arr[t_idx]
        left = np.searchsorted(tt, s_arr[a_idx], side="left"); right = np.searchsorted(tt, e_arr[a_idx], side="right")
        counts = right - left; valid = counts > 0
        if not valid.any(): continue
        va, vc, vl = a_idx[valid], counts[valid], left[valid]
        a_chunks.append(np.repeat(va, vc))
        ends = np.cumsum(vc); starts = np.concatenate([[0], ends[:-1]])
        within = np.arange(int(vc.sum())) - np.repeat(starts, vc)
        t_chunks.append(t_idx[np.repeat(vl, vc) + within])
    if not a_chunks:
        raise ValueError("no anchor/target pairs")
    ap, tp = np.concatenate(a_chunks), np.concatenate(t_chunks)
    merged = df.take(order[ap]); merged.index = pd.RangeIndex(len(merged))
    for col in ["TRADE_DATE", f"TRADE_DATE_PLUS_{trading_days}", "TARGET_ANCHOR_TIME", "TARGET_ANCHOR_GAP_HOURS", "_ANCHOR_ID"]:
        merged[col] = out[col].take(ap).set_axis(merged.index)
    tf = targets.take(tp); tf.index = merged.index
    for col in target_cols:
        if col != "CUSIP":
            merged[f"{col}_NEXT"] = tf[col]
    idx_next = pd.MultiIndex.from_frame(merged[["BM_CUSIP_NEXT", "DATE_NEXT"]]); idx_curr = pd.MultiIndex.from_frame(merged[["BM_CUSIP", "DATE_NEXT"]])
    bm_next = pd.Series(idx_next.map(bm_dict), index=merged.index); bm_curr = pd.Series(idx_curr.map(bm_dict), index=merged.index)
    bm_changed = merged["BM_CUSIP"].ne(merged["BM_CUSIP_NEXT"]) & merged["BM_CUSIP_NEXT"].notna() & merged["BM_CUSIP"].notna()
    merged["D_BM_YIELD_OFFSET_TARGET"] = np.where(bm_changed, (bm_next - bm_curr).fillna(0.0), 0.0)
    merged["TARGET"] = (merged["BM_SPREAD_NEXT"] - merged["BM_SPREAD"]) + merged["D_BM_YIELD_OFFSET_TARGET"]
    merged["TARGET_GAP_HOURS"] = (merged["EFFECTIVE_DATETIME_TS_NEXT"] - merged["EFFECTIVE_DATETIME_TS"]).dt.total_seconds() / 3600.0
    merged["TARGET_WINDOW_TRADE_COUNT"] = merged.groupby("_ANCHOR_ID", sort=False)["_ANCHOR_ID"].transform("size")
    del merged["_ANCHOR_ID"]
    return merged

pairs = attach_forward_target_single_horizon(base_df, bm_dict, trading_days=1)
print(f"{len(pairs):,} anchor-target pairs from {pairs['TRADE_ROW_ID'].nunique():,} anchors; |TARGET| mean {100 * pairs.TARGET.abs().mean():.3f} bps, median {100 * pairs.TARGET.abs().median():.3f}")

# ---- pair-level features: old (TRADE_TYPE) and new (EFF_SIDE) target-side quote deviations ------------------------------------
_SGN = {"B": 1.0, "S": -1.0, "D": 0.0}
def target_side_quote(side_col):
    s = pairs[side_col].astype(str)
    return np.select([s.eq("B").to_numpy(), s.eq("S").to_numpy()], [pairs["BID_SPREAD_CPP"].to_numpy(), pairs["ASK_SPREAD_CPP"].to_numpy()], pairs["MID_SPREAD_CPP"].to_numpy())
pairs["CPP_TARGET_SIDE_DEV"] = target_side_quote("TRADE_TYPE_NEXT") - pairs["BM_SPREAD"].to_numpy()
pairs["CPP_TARGET_SIDE_DEV_EFF"] = target_side_quote("EFF_SIDE_NEXT") - pairs["BM_SPREAD"].to_numpy()
pairs["CPP_TARGET_SIDE_DEV_NORM"] = (pairs["CPP_TARGET_SIDE_DEV"] / pairs["CPP_WIDTH"]).astype("float32")
pairs["CPP_TARGET_SIDE_DEV_EFF_NORM"] = (pairs["CPP_TARGET_SIDE_DEV_EFF"] / pairs["CPP_WIDTH"]).astype("float32")
pairs["SIDE_FLIP"] = (pairs["TRADE_TYPE_NEXT"].astype(str).map(_SGN).fillna(0.0) - pairs["TRADE_TYPE"].astype(str).map(_SGN).fillna(0.0)).astype("float32")
pairs["SIDE_FLIP_EFF"] = (pairs["EFF_SIDE_NEXT"].astype(str).map(_SGN).fillna(0.0) - pairs["EFF_SIDE"].astype(str).map(_SGN).fillna(0.0)).astype("float32")
pairs["EXPECTED_BOUNCE"] = (pairs["SIDE_FLIP"] * pairs["CPP_WIDTH"] / 2.0).astype("float32")
pairs["EXPECTED_BOUNCE_EFF"] = (pairs["SIDE_FLIP_EFF"] * pairs["CPP_WIDTH"] / 2.0).astype("float32")
pairs["IS_OVER_WEEKEND"] = (pairs["TARGET_ANCHOR_GAP_HOURS"] > 30).astype("float32")
pairs["IS_BURST_NEXT"] = pairs["IS_BURST_NEXT"].astype("float32"); pairs["IS_EOD_NEXT"] = pairs["IS_EOD_NEXT"].astype("float32"); pairs["IS_SESSION_NEXT"] = pairs["IS_SESSION_NEXT"].astype("float32")
pairs["TARGET_CLASS"] = np.select([pairs.IS_SESSION_NEXT.eq(1), pairs.IS_BURST_NEXT.eq(1) & pairs.IS_EOD_NEXT.eq(1), pairs.IS_BURST_NEXT.eq(1), pairs.IS_EOD_NEXT.eq(1)],
                                  ["interdealer session", "EOD burst", "intraday burst", "EOD single"], "RFQ-like")

# ---- exact decomposition: TARGET = cpp_side_eff + CPP_DRIFT + TARGET_PRINT_NOISE ----------------------------------------------
_q_u = np.select([pairs["EFF_SIDE_NEXT"].astype(str).eq("B").to_numpy(), pairs["EFF_SIDE_NEXT"].astype(str).eq("S").to_numpy()],
                 [pairs["BID_SPREAD_CPP_NEXT"].to_numpy(), pairs["ASK_SPREAD_CPP_NEXT"].to_numpy()], pairs["MID_SPREAD_CPP_NEXT"].to_numpy())
pairs["CPP_DRIFT"] = (_q_u - target_side_quote("EFF_SIDE_NEXT") + pairs["D_BM_YIELD_OFFSET_TARGET"].to_numpy()).astype("float64")
pairs["TARGET_PRINT_NOISE"] = (pairs["BM_SPREAD_NEXT"].to_numpy() - _q_u).astype("float64")
_chk = (pairs["CPP_TARGET_SIDE_DEV_EFF"] + pairs["CPP_DRIFT"] + pairs["TARGET_PRINT_NOISE"] - pairs["TARGET"]).abs().max()
print(f"decomposition identity check: max |cpp_side_eff + drift + noise - TARGET| = {_chk:.2e}")

# mid-quote drift split: anchor -> last print of bond on T, -> first print on T+1, -> target
_bd = base_df.sort_values("EFFECTIVE_DATETIME_TS").groupby(["CUSIP", "DATE"], sort=False)["MID_SPREAD_CPP"].agg(["first", "last"]).reset_index()
pairs = pairs.merge(_bd.rename(columns={"first": "_OPEN_MID_T", "last": "_CLOSE_MID_T"}), on=["CUSIP", "DATE"], how="left")
pairs = pairs.merge(_bd.rename(columns={"DATE": "DATE_NEXT", "first": "_OPEN_MID_T1", "last": "_CLOSE_MID_T1"}), on=["CUSIP", "DATE_NEXT"], how="left")
pairs["DRIFT_REST_OF_T"] = (pairs["_CLOSE_MID_T"] - pairs["MID_SPREAD_CPP"]).astype("float32")
pairs["DRIFT_OVERNIGHT"] = (pairs["_OPEN_MID_T1"] - pairs["_CLOSE_MID_T"] + pairs["D_BM_YIELD_OFFSET_TARGET"]).astype("float32")
pairs["DRIFT_DAY_T1"] = (pairs["MID_SPREAD_CPP_NEXT"] - pairs["_OPEN_MID_T1"]).astype("float32")
pairs["CPP_MID_DRIFT"] = (pairs["MID_SPREAD_CPP_NEXT"] - pairs["MID_SPREAD_CPP"] + pairs["D_BM_YIELD_OFFSET_TARGET"]).astype("float32")
pairs = pairs.drop(columns=["_OPEN_MID_T", "_CLOSE_MID_T", "_OPEN_MID_T1", "_CLOSE_MID_T1"])
for c in ["CPP_TARGET_SIDE_DEV", "CPP_TARGET_SIDE_DEV_EFF"]:
    pairs[c] = pairs[c].astype("float64")
print(f"pair frame {pairs.memory_usage(deep=False).sum() / 1e9:.2f} GB, {pairs.shape[1]} columns; target class mix: {pairs.TARGET_CLASS.value_counts(normalize=True).round(3).to_dict()}")
print(f"target side: TRADE_TYPE_NEXT == 'D' on {pairs.TRADE_TYPE_NEXT.astype(str).eq('D').mean():.1%} of pairs, of which EFF_SIDE_NEXT is customer-sided on "
      f"{(pairs.TRADE_TYPE_NEXT.astype(str).eq('D') & pairs.EFF_SIDE_NEXT.astype(str).ne('D')).sum() / max(pairs.TRADE_TYPE_NEXT.astype(str).eq('D').sum(), 1):.1%}")

## 4. Where is the gap? The CPP-drift decomposition

Target-equal statistics over all pairs. `cpp_side_eff` errs by (drift + noise); the "CPP-at-target oracle" `cpp_side_eff + CPP_DRIFT` errs only by the target-print noise and is the ceiling for any model that does not see the target print. The three drift pieces show whether the drift happens on the anchor day after the anchor (partly predictable: the composite catching up with today's prints and flow), overnight (news), or on the target day before the target (news and flow that has not happened yet).

The binned plot of `CPP_DRIFT` against `cpp_side_eff` is the mechanism behind $\beta < 1$: if the composite moves toward the anchor print by tomorrow, the slope of drift on the deviation is $-(1-\beta)$; the by-print-class version shows whether the same $\beta$ should apply to a portfolio-trade line and to a 5MM voice RFQ.

In [ ]:
P = pairs
terms = pd.DataFrame({
    "MAE if this term were the only error (bps)": {
        "cpp_side_eff error = drift + noise": bps(P.TARGET - P.CPP_TARGET_SIDE_DEV_EFF).abs().mean(),
        "cpp_side (TRADE_TYPE) error": bps(P.TARGET - P.CPP_TARGET_SIDE_DEV).abs().mean(),
        "CPP drift only (side quote)": bps(P.CPP_DRIFT).abs().mean(),
        "target-print noise only  = CPP-at-target oracle": bps(P.TARGET_PRINT_NOISE).abs().mean(),
        "no-change (dummy)": bps(P.TARGET).abs().mean(),
    }})
print(terms.round(3).to_string())
split = pd.DataFrame({"mean_abs_bps": bps(P[["DRIFT_REST_OF_T", "DRIFT_OVERNIGHT", "DRIFT_DAY_T1", "CPP_MID_DRIFT"]]).abs().mean(),
                      "std_bps": bps(P[["DRIFT_REST_OF_T", "DRIFT_OVERNIGHT", "DRIFT_DAY_T1", "CPP_MID_DRIFT"]]).std(),
                      "coverage": P[["DRIFT_REST_OF_T", "DRIFT_OVERNIGHT", "DRIFT_DAY_T1", "CPP_MID_DRIFT"]].notna().mean()})
_v = bps(P[["DRIFT_REST_OF_T", "DRIFT_OVERNIGHT", "DRIFT_DAY_T1"]]).dropna()
split.loc[["DRIFT_REST_OF_T", "DRIFT_OVERNIGHT", "DRIFT_DAY_T1"], "variance_share_%"] = 100 * _v.var() / _v.sum(axis=1).var()
print("\nmid-quote drift split (anchor -> close T -> open T+1 -> target):")
print(split.round(3).to_string())
# how much of |drift| is explained by each piece given the others? (variance shares add up to ~100 with covariance)
_cov = _v.cov(); print(f"\ncorr between pieces:\n{_v.corr().round(3).to_string()}")
terms.to_csv(OUT_DIR / "decomp_terms.csv"); split.to_csv(OUT_DIR / "decomp_drift_split.csv")

# beta mechanism: binned drift vs cpp_side_eff, by print class of the anchor
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax = axes[0]
terms.iloc[:, 0].sort_values().plot.barh(ax=ax, color=sns.color_palette("crest", 5)); ax.set_xlabel("MAE (bps)"); ax.set_title("error terms of the decomposition")
ax = axes[1]
_x = bps(P.CPP_TARGET_SIDE_DEV_EFF).clip(-15, 15); _y = bps(P.CPP_DRIFT).clip(-30, 30)
for cls_, g in P.assign(x=_x, y=_y).groupby("PRINT_CLASS", observed=True):
    if len(g) < 2000: continue
    b_ = pd.qcut(g.x, 15, duplicates="drop"); prof = g.groupby(b_, observed=True).agg(x=("x", "mean"), y=("y", "mean"))
    sl = ols_slope(g.x, g.y)[0]
    ax.plot(prof.x, prof.y, marker="o", ms=3, label=f"{cls_} (slope {sl:+.2f}, implied beta {1 + sl:.2f})")
ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("cpp_side_eff = target-side CPP quote - anchor print (bps)"); ax.set_ylabel("realised CPP drift to target (bps)")
ax.set_title("does the composite move toward the anchor print by T+1?"); ax.legend(fontsize=8)
ax = axes[2]
_d = bps(P[["DRIFT_REST_OF_T", "DRIFT_OVERNIGHT", "DRIFT_DAY_T1"]]).clip(-10, 10)
for c in _d.columns:
    sns.kdeplot(_d[c].dropna().sample(min(len(_d), 200_000), random_state=SEED), ax=ax, label=f"{c} (mean|.| {split.loc[c, 'mean_abs_bps']:.2f})", lw=1.5)
ax.set_xlim(-6, 6); ax.set_xlabel("bps"); ax.set_title("distribution of the three drift pieces"); ax.legend(fontsize=8)
savefig("decomp_4_terms_and_beta")

# market-level component of drift by target date and its predictability from the previous day
dd = P.groupby("DATE_NEXT").agg(mkt_drift=("CPP_MID_DRIFT", "mean"), mkt_target=("TARGET", "mean"), n=("TARGET", "size"))
dd["mkt_drift_prev"] = dd["mkt_drift"].shift(1)
print(f"\nday-level mean CPP drift: std {100 * dd.mkt_drift.std():.3f} bps; lag-1 autocorr {dd.mkt_drift.corr(dd.mkt_drift_prev):+.3f}; "
      f"corr(day mean drift, day mean TARGET) {dd.mkt_drift.corr(dd.mkt_target):+.3f}")

## 5. Walk-forward evaluation

Same protocol as the companion notebook: expanding train window, 20 validation days for early stopping and calibration, 20 test days, 2-day embargo, three folds (auto-scaled in `SAMPLE_MODE`); target-equal MAE in bps; paired-by-test-day $t$ statistics. Every rung is fitted on the fold's train dates only.

### 5.1 Folds, helpers and feature screening

Before any model, each new feature is screened three ways on the whole tape: correlation with `TARGET`, with the residual of `cpp_side_eff x beta` ($\beta$ fixed at the companion's 0.67 for screening), and with `CPP_DRIFT`. A feature that correlates with the residual and with drift but not with the raw target is exactly what a drift model needs and what the noisy per-print target hides. The small-multiples plot shows the mean residual by feature decile for the strongest features.

In [ ]:
unique_dates = sorted(pairs["DATE"].dropna().unique()); n_dates = len(unique_dates)
if n_dates >= 150:
    min_train_days, val_days, test_days, step_days, n_folds, embargo = 101, 20, 20, 20, 3, 2
else:
    n_folds = 2; val_days = test_days = step_days = max(2, n_dates // 8); embargo = 1
    min_train_days = n_dates - (val_days + test_days + (n_folds - 1) * step_days)
windows = []
for fold_idx in range(n_folds):
    tr_end = min_train_days + fold_idx * step_days
    va_end = min(tr_end + val_days, n_dates); te_end = min(va_end + test_days, n_dates)
    tr = list(unique_dates[:max(0, tr_end - embargo)]); va = list(unique_dates[tr_end:max(tr_end, va_end - embargo)]); te = list(unique_dates[va_end:te_end])
    if tr and va and te:
        windows.append((fold_idx + 1, tr, va, te))
print(f"{n_dates} dates -> {len(windows)} folds: " + "; ".join(f"fold {f}: train {len(a)} / val {len(b)} / test {len(c)} days" for f, a, b, c in windows))

def _wmae(frame, col, y="y_true"):
    e = (pd.to_numeric(frame[col], errors="coerce") - frame[y]).abs() * 100.0
    return float(e.mean()) if e.notna().any() else np.nan

def _paired_days(frame, col_a, col_b, y="y_true"):
    ea = (pd.to_numeric(frame[col_a], errors="coerce") - frame[y]).abs() * 100.0
    eb = (pd.to_numeric(frame[col_b], errors="coerce") - frame[y]).abs() * 100.0
    v = ea.notna() & eb.notna()
    d = pd.DataFrame({"a": ea[v], "b": eb[v], "DATE": frame.loc[v, "DATE"]}).groupby("DATE").mean()
    diff = d["b"] - d["a"]
    t = diff.mean() / diff.std(ddof=1) * np.sqrt(len(diff)) if len(diff) > 1 and diff.std(ddof=1) > 0 else np.nan
    return len(diff), float(diff.mean()), float(t), float((diff > 0).mean())

def _lad_fit(X, y, w=None, n_iter=15, eps=1e-4, max_rows=1_000_000, seed=0, ridge=0.0):
    #   weighted LAD by IRLS (companion notebook); optional ridge penalty (scale-free: lambda = ridge * n) on all columns but the first (intercept)
    X, y = np.asarray(X, float), np.asarray(y, float)
    w = np.ones(len(y)) if w is None else np.asarray(w, float)
    ok = np.isfinite(X).all(axis=1) & np.isfinite(y) & np.isfinite(w); X, y, w = X[ok], y[ok], w[ok]
    if len(y) == 0: return np.full(X.shape[1], np.nan)
    if len(y) > max_rows:
        idx = np.random.default_rng(seed).choice(len(y), max_rows, replace=False); X, y, w = X[idx], y[idx], w[idx]
    k = X.shape[1]
    if ridge > 0:
        R = np.sqrt(ridge * len(y)) * np.eye(k); R[0, 0] = 0.0
        Xa, ya = np.vstack([X, R]), np.concatenate([y, np.zeros(k)])
    def solve(sw):
        sw_a = np.concatenate([sw, np.ones(k)]) if ridge > 0 else sw
        A, b = (Xa, ya) if ridge > 0 else (X, y)
        return np.linalg.lstsq(A * sw_a[:, None], b * sw_a, rcond=None)[0]
    beta = solve(np.sqrt(w))
    for _ in range(n_iter):
        ww = w / np.maximum(np.abs(y - X @ beta), eps)
        beta = solve(np.sqrt(ww))
    return beta

# ---- feature screening ----------------------------------------------------------------------------------------------------
NEW_NUMERIC = [c for c in SIDE_FEATURES + CPPBIAS_FEATURES + MULTIDAY_FEATURES + TIME_FEATURES if c != "EFF_SIDE" and c in pairs.columns]
_res = pairs["TARGET"] - 0.67 * pairs["CPP_TARGET_SIDE_DEV_EFF"]
_u = pairs.drop_duplicates("TRADE_ROW_ID")   # one row per anchor for anchor-level correlations
_res_u = _res.loc[_u.index]
scr = pd.DataFrame({
    "corr_TARGET": _u[NEW_NUMERIC].apply(pd.to_numeric, errors="coerce").corrwith(_u["TARGET"]),
    "corr_resid_cpp_beta": _u[NEW_NUMERIC].apply(pd.to_numeric, errors="coerce").corrwith(_res_u),
    "corr_CPP_DRIFT": _u[NEW_NUMERIC].apply(pd.to_numeric, errors="coerce").corrwith(_u["CPP_DRIFT"]),
    "coverage": _u[NEW_NUMERIC].notna().mean(),
})
scr["block"] = ["SIDE" if c in SIDE_FEATURES else "CPPBIAS" if c in CPPBIAS_FEATURES else "MULTIDAY" if c in MULTIDAY_FEATURES else "TIME" for c in scr.index]
scr = scr.sort_values("corr_resid_cpp_beta", key=np.abs, ascending=False)
print("\nscreening of new anchor-level features (one row per anchor):")
print(scr.round(3).to_string())
scr.to_csv(OUT_DIR / "feature_screening.csv")

top = [c for c in scr.index if scr.loc[c, "coverage"] > 0.3][:8]
fig, axes = plt.subplots(2, 4, figsize=(18, 7.5)); axes = axes.ravel()
for ax, c in zip(axes, top):
    x = pd.to_numeric(_u[c], errors="coerce")
    try:
        b_ = pd.qcut(x, 10, duplicates="drop")
    except ValueError:
        b_ = x
    prof = pd.DataFrame({"resid": bps(_res_u), "drift": bps(_u["CPP_DRIFT"])}).groupby(b_.to_numpy(), observed=True).mean()
    ax.plot(range(len(prof)), prof["resid"], marker="o", label="residual of cpp_side_eff x 0.67"); ax.plot(range(len(prof)), prof["drift"], marker="s", ms=3, label="CPP drift", alpha=0.8)
    ax.axhline(0, color="k", lw=0.7); ax.set_title(f"{c}\n({scr.loc[c, 'block']})", fontsize=9); ax.set_xlabel("decile")
axes[0].legend(fontsize=7); axes[0].set_ylabel("mean (bps)")
plt.suptitle("mean residual / CPP drift by feature decile (all pairs, one row per anchor)", y=1.02)
savefig("screen_5_1_residual_by_decile")

### 5.2 Ladder of simple rules: from `cpp_side` to a print-quality-aware $\beta$ and a small LAD model

| rung | parameters | what it tests |
| --- | --- | --- |
| `dummy` | 0 | difficulty |
| `cpp_side` | 0 | the companion's production benchmark (`TRADE_TYPE`-based side) |
| `cpp_side_eff` | 0 | the same quote picked with `EFF_SIDE` for anchor and target: pure label hygiene |
| `cpp_side x beta`, `cpp_side_eff x beta` | 1 each | one shrinkage coefficient, LAD on train dates |
| `quality beta` | ~12 | $\beta$ as a linear function of anchor print-quality flags: interdealer, mislabeled-D, markup, ATS, burst, session, EOD, riskless-principal leg, size z-score, morning anchor, plus an intercept |
| `lad-plus` | ~26 | quality beta + linear terms for the CPP-bias/lag block and the strongest multi-day features |

Each fitted rung is scored on the fold's test dates; the table reports MAE, per-fold MAE, incremental gain over the previous rung and the paired-day $t$ against `cpp_side_eff x beta`, the natural new reference.

In [ ]:
QUALITY_FLAGS = ["IS_TRUE_DD", "IS_MISLABELED_D", "IS_MARKUP", "IS_ATS", "IS_BURST", "IS_SESSION", "IS_EOD", "IS_RP_D_LEG", "IS_MORNING_ANCHOR"]
#   MKT_DEV_EFF_PREV_D is deliberately excluded from the linear rung: it is constant within a day and a LAD fit on ~100 days would use it as a day label.
LADPLUS_LIN = [c for c in ["PREV_DEV_EFF", "ROLL_DEV_EFF_5P", "CUSIP_DEV_EFF_PREV_5D", "ISS_DEV_EFF_TODAY", "ISS_DEV_EFF_PREV_D", "MKT_DEV_EFF_3H",
                           "CPP_MID_CHG_SINCE_PREV", "CPP_MID_CHG_24H", "SESSION_DEV_MED", "FLOW_5D", "RV_5D_NET", "ISS_CURVE_RESID", "EXPECTED_BOUNCE_EFF"] if c in pairs.columns]
LADPLUS_RIDGE = 1e-3

def _qz(frame):
    return ((pd.to_numeric(frame["LOG_QTY"], errors="coerce").fillna(5.7) - 5.7) / 0.6).clip(-3, 3).to_numpy()

def design_quality(frame):
    x = frame["CPP_TARGET_SIDE_DEV_EFF"].to_numpy(float)
    cols = [np.ones(len(frame)), x] + [x * pd.to_numeric(frame[f], errors="coerce").fillna(0).to_numpy(float) for f in QUALITY_FLAGS] + [x * _qz(frame)]
    return np.column_stack(cols)
QUALITY_NAMES = ["intercept", "cpp_side_eff"] + [f"cpp_side_eff x {f}" for f in QUALITY_FLAGS] + ["cpp_side_eff x size_z"]

_scale = {}
def design_ladplus(frame, fit=False):
    #   linear terms are winsorised at the train 0.5/99.5 percentiles, standardised with train mean/std, and NaN -> 0 (= train mean)
    parts = [design_quality(frame)]
    for c in LADPLUS_LIN:
        v = pd.to_numeric(frame[c], errors="coerce")
        if fit:
            lo, hi = v.quantile(0.005), v.quantile(0.995); vc = v.clip(lo, hi)
            _scale[c] = (lo, hi, float(vc.mean()), float(vc.std()) or 1.0)
        lo, hi, mu, sd = _scale[c]
        parts.append(((v.clip(lo, hi) - mu) / sd).fillna(0.0).to_numpy(float)[:, None])
    return np.column_stack(parts)
LADPLUS_NAMES = QUALITY_NAMES + LADPLUS_LIN

EVAL_COLS = ["TRADE_ROW_ID", "TRADE_ROW_ID_NEXT", "CUSIP", "ISSUER", "DATE", "DATE_NEXT", "TARGET", "CPP_TARGET_SIDE_DEV", "CPP_TARGET_SIDE_DEV_EFF", "CPP_DRIFT",
             "TARGET_PRINT_NOISE", "TARGET_WINDOW_TRADE_COUNT", "PRINT_CLASS", "TARGET_CLASS", "EFF_SIDE", "EFF_SIDE_NEXT", "TRADE_TYPE", "TRADE_TYPE_NEXT",
             "IS_MISLABELED_D", "QUANTITY", "HOURS_SINCE_PREV_TRADE", "CPP_WIDTH", "TARGET_ANCHOR_GAP_HOURS", "LOG_QTY", *QUALITY_FLAGS, *LADPLUS_LIN]
EVAL_COLS = [c for c in dict.fromkeys(EVAL_COLS) if c in pairs.columns]
ev = pairs.loc[pairs["DATE"].isin(sum([w[2] + w[3] for w in windows], [])), EVAL_COLS].copy()
ev["y_true"] = ev["TARGET"].astype(float)
ev["fold_id"] = 0; ev["sample"] = ""
for fold_id, _, va, te in windows:
    ev.loc[ev.DATE.isin(va), ["fold_id", "sample"]] = [fold_id, "val"]; ev.loc[ev.DATE.isin(te), ["fold_id", "sample"]] = [fold_id, "test"]
ev = ev[ev["sample"].ne("")].reset_index(drop=True)
ev["pred_dummy"] = 0.0; ev["pred_cpp_side"] = ev["CPP_TARGET_SIDE_DEV"]; ev["pred_cpp_side_eff"] = ev["CPP_TARGET_SIDE_DEV_EFF"]
for c in ["pred_cpp_beta", "pred_cpp_eff_beta", "pred_quality_beta", "pred_ladplus"]:
    ev[c] = np.nan

coef_rows = []
for fold_id, tr_dates, _, _ in windows:
    trn = pairs.loc[pairs["DATE"].isin(tr_dates)]
    y = trn["TARGET"].to_numpy(float)
    b_tt = _lad_fit(trn[["CPP_TARGET_SIDE_DEV"]].to_numpy(float), y, seed=fold_id)
    b_ef = _lad_fit(trn[["CPP_TARGET_SIDE_DEV_EFF"]].to_numpy(float), y, seed=fold_id)
    b_q = _lad_fit(design_quality(trn), y, seed=fold_id)
    b_lp = _lad_fit(design_ladplus(trn, fit=True), y, seed=fold_id, ridge=LADPLUS_RIDGE)
    m = ev["fold_id"].eq(fold_id).to_numpy()
    e = ev.loc[m]
    ev.loc[m, "pred_cpp_beta"] = b_tt[0] * e["CPP_TARGET_SIDE_DEV"].to_numpy(float)
    ev.loc[m, "pred_cpp_eff_beta"] = b_ef[0] * e["CPP_TARGET_SIDE_DEV_EFF"].to_numpy(float)
    ev.loc[m, "pred_quality_beta"] = design_quality(e) @ b_q
    ev.loc[m, "pred_ladplus"] = design_ladplus(e) @ b_lp
    coef_rows.append(pd.Series({"beta_cpp_side": b_tt[0], "beta_cpp_side_eff": b_ef[0], **dict(zip(QUALITY_NAMES, b_q))}, name=f"fold{fold_id}"))
    coef_rows[-1] = pd.concat([coef_rows[-1], pd.Series(dict(zip([f"ladplus:{n}" for n in LADPLUS_NAMES], b_lp)), name=f"fold{fold_id}")])
coefs = pd.concat(coef_rows, axis=1)
_show = coefs.loc[[i for i in coefs.index if not i.startswith("ladplus:") or i.split(":")[1] in LADPLUS_LIN]].copy()
_lin = _show.index.str.startswith("ladplus:")
_show.loc[_lin] = _show.loc[_lin] * 100
_show.index = [i.replace("ladplus:", "lad-plus, bps per 1 train-sd: ") if i.startswith("ladplus:") else i for i in _show.index]
print("fitted coefficients by fold. quality rule: implied beta for a print = cpp_side_eff coefficient + sum of its active flag coefficients;")
print("lad-plus linear terms are in bps of TARGET per one train-sd move of the (winsorised) feature:")
print(_show.round(3).to_string())
coefs.to_csv(OUT_DIR / "ladder_coefficients.csv")

LADDER = [("0 dummy", "pred_dummy"), ("1 cpp_side (TRADE_TYPE)", "pred_cpp_side"), ("1e cpp_side_eff (EFF_SIDE)", "pred_cpp_side_eff"),
          ("2 cpp_side x beta", "pred_cpp_beta"), ("2e cpp_side_eff x beta", "pred_cpp_eff_beta"),
          (f"3 quality beta ({len(QUALITY_NAMES)} params)", "pred_quality_beta"), (f"4 lad-plus ({len(LADPLUS_NAMES)} params)", "pred_ladplus")]
REF = "pred_cpp_eff_beta"
te = ev[ev["sample"].eq("test")]
def ladder_table(frame, rungs, ref):
    rows = []
    for name, col in rungs:
        if col not in frame.columns or frame[col].notna().mean() < 0.5: continue
        r = {"rung": name, "test_mae": _wmae(frame, col)}
        for f, g in frame.groupby("fold_id"):
            r[f"fold{f}"] = _wmae(g, col)
        r["gain_%_vs_dummy"] = 100 * (1 - r["test_mae"] / _wmae(frame, "pred_dummy"))
        r["gain_%_vs_ref"] = 100 * (1 - r["test_mae"] / _wmae(frame, ref))
        r["worst_fold_%_vs_ref"] = min(100 * (1 - r[f"fold{f}"] / _wmae(g, ref)) for f, g in frame.groupby("fold_id"))
        _, _, r["t_vs_ref"], r["days_win_vs_ref"] = _paired_days(frame, col, ref)
        rows.append(r)
    return pd.DataFrame(rows).set_index("rung")
ladder = ladder_table(te, LADDER, REF)
print(f"\nT+1 ladder, target-equal test MAE (bps); reference = cpp_side_eff x beta:")
print(ladder.round(3).to_string())
ladder.to_csv(OUT_DIR / "ladder_simple.csv")
_, _, t_eff, w_eff = _paired_days(te, "pred_cpp_side_eff", "pred_cpp_side")
print(f"\nlabel hygiene alone: cpp_side_eff vs cpp_side gain {100 * (1 - _wmae(te, 'pred_cpp_side_eff') / _wmae(te, 'pred_cpp_side')):+.2f}%, paired-day t = {t_eff:.2f}, wins {w_eff:.0%} of days")

### 5.3 LightGBM: old core, old core + new blocks, and block ablation

`fit_lgbm` trains one model per fold on train dates with early stopping on validation dates (MAE objective, same regularisation as the companion), with the interaction constraint that regime-like features only interact with bond-level features. Models:

- `lgbm-old-core`: the compact reproduction of the companion's features, with the companion's `TRADE_TYPE`-based pair features.
- `lgbm-old+SIDE`, `+CPPBIAS`, `+MULTIDAY`, `+TIME`: one block added at a time (ablation, `RUN_ABLATION`).
- `lgbm-all`: old core + all blocks + `EFF_SIDE`-based pair features.

`SIDE_PAIR_PRIOR` is filled from train dates per fold, as in the companion. Training rows can be capped with `LGBM_TRAIN_ROW_CAP` for runtime.

In [ ]:
PAIR_OLD = ["TRADE_TYPE_NEXT", "CPP_TARGET_SIDE_DEV", "CPP_TARGET_SIDE_DEV_NORM", "SIDE_FLIP", "EXPECTED_BOUNCE", "TARGET_ANCHOR_GAP_HOURS", "SIDE_PAIR_PRIOR"]
PAIR_NEW = ["EFF_SIDE_NEXT", "CPP_TARGET_SIDE_DEV_EFF", "CPP_TARGET_SIDE_DEV_EFF_NORM", "SIDE_FLIP_EFF", "EXPECTED_BOUNCE_EFF", "IS_BURST_NEXT", "IS_EOD_NEXT", "IS_OVER_WEEKEND", "SIDE_PAIR_PRIOR_EFF"]
BLOCKS = {"SIDE": SIDE_FEATURES, "CPPBIAS": CPPBIAS_FEATURES, "MULTIDAY": MULTIDAY_FEATURES, "TIME": TIME_FEATURES}
FEATURE_SETS = {
    "lgbm-old-core": OLD_CORE + PAIR_OLD,
    "lgbm-all": OLD_CORE + PAIR_OLD + PAIR_NEW + sum(BLOCKS.values(), []),
}
if RUN_ABLATION:
    for k, v in BLOCKS.items():
        FEATURE_SETS[f"lgbm-old+{k}"] = OLD_CORE + PAIR_OLD + v + (PAIR_NEW if k == "SIDE" else [])
FEATURE_SETS = {k: [c for c in dict.fromkeys(v) if c in pairs.columns or c in ("SIDE_PAIR_PRIOR", "SIDE_PAIR_PRIOR_EFF")] for k, v in FEATURE_SETS.items()}

for c in CAT_FEATURES:
    if c in pairs.columns:
        pairs[c] = pairs[c].astype(str).astype("category")
pairs["SIDE_PAIR_PRIOR"] = np.float32(np.nan); pairs["SIDE_PAIR_PRIOR_EFF"] = np.float32(np.nan)

LGB_PARAMS = dict(objective="mae", boosting_type="gbdt", learning_rate=0.05, n_estimators=2000, num_leaves=31, max_depth=6, min_child_samples=80,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_alpha=0.2, reg_lambda=1.5, max_bin=255, n_jobs=-1, verbosity=-1, random_state=42)
EARLY_STOP = 100

def _fill_priors(tr_mask):
    for side_a, side_u, col in [("TRADE_TYPE", "TRADE_TYPE_NEXT", "SIDE_PAIR_PRIOR"), ("EFF_SIDE", "EFF_SIDE_NEXT", "SIDE_PAIR_PRIOR_EFF")]:
        pr = pairs.loc[tr_mask].groupby([side_a, side_u], observed=True)["TARGET"].mean().rename(col).reset_index()
        pairs[col] = pairs[[side_a, side_u]].merge(pr, on=[side_a, side_u], how="left", sort=False)[col].to_numpy(dtype="float32")

def _params_for(feats):
    p = dict(LGB_PARAMS)
    reg = [i for i, f in enumerate(feats) if f in REGIME_LIKE]; bond = [i for i, f in enumerate(feats) if f not in REGIME_LIKE]
    if reg:
        p["interaction_constraints"] = [bond + [i] for i in reg]
    return p

def fit_lgbm(name, feats, target="TARGET", out_col=None, row_cap=LGBM_TRAIN_ROW_CAP, seed=SEED):
    out_col = out_col or f"pred_{name}"
    ev[out_col] = np.nan
    cats = [c for c in CAT_FEATURES if c in feats]
    iters, models = [], []
    for fold_id, tr_dates, va_dates, te_dates in tqdm(windows, desc=name, leave=False):
        tr = pairs["DATE"].isin(tr_dates).to_numpy(); va = pairs["DATE"].isin(va_dates).to_numpy()
        _fill_priors(tr)
        tr_idx = np.flatnonzero(tr)
        if row_cap is not None and len(tr_idx) > row_cap:
            tr_idx = np.random.default_rng(seed + fold_id).choice(tr_idx, row_cap, replace=False)
        y_tr = pairs[target].to_numpy(float)[tr_idx]; ok = np.isfinite(y_tr)
        y_va = pairs.loc[va, target].to_numpy(float); okv = np.isfinite(y_va)
        mdl = lgb.LGBMRegressor(**_params_for(feats))
        mdl.fit(pairs.iloc[tr_idx[ok]][feats], y_tr[ok], eval_set=[(pairs.loc[va].iloc[np.flatnonzero(okv)][feats], y_va[okv])], eval_metric="l1",
                callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOP, verbose=False)], categorical_feature=cats)
        iters.append(int(getattr(mdl, "best_iteration_", None) or mdl.n_estimators)); models.append(mdl)
        for sample, dates in [("val", va_dates), ("test", te_dates)]:
            mk = pairs["DATE"].isin(dates)
            pred = pd.Series(mdl.predict(pairs.loc[mk, feats]), index=pairs.index[mk])
            em = ev["fold_id"].eq(fold_id) & ev["sample"].eq(sample)
            key = pairs.loc[mk, ["TRADE_ROW_ID", "TRADE_ROW_ID_NEXT"]].assign(p=pred.to_numpy())
            ev.loc[em, out_col] = ev.loc[em, ["TRADE_ROW_ID", "TRADE_ROW_ID_NEXT"]].merge(key, on=["TRADE_ROW_ID", "TRADE_ROW_ID_NEXT"], how="left")["p"].to_numpy()
    print(f"{name}: {len(feats)} features, best_iter {iters}, test coverage {ev.loc[ev['sample'].eq('test'), out_col].notna().mean():.1%}, elapsed {(time.time() - t_start) / 60:.1f} min")
    return models

LGBM_MODELS = {}
if RUN_LGBM:
    for name, feats in FEATURE_SETS.items():
        LGBM_MODELS[name] = fit_lgbm(name, feats)

### 5.4 Results: full ladder, ablation, and where the gains sit

The full ladder adds the LightGBM rungs to the simple rules. The ablation bars show the MAE change from adding each block to the old core, with per-fold markers. The split tables score every rung separately by target class (RFQ-like vs burst / EOD) and by anchor class, because production use is RFQ pricing while a large share of the backtest pairs are portfolio-trade lines, and by the anchor's `TRADE_TYPE == 'D'` mislabel flag, where the side fix should show up most.

In [ ]:
te = ev[ev["sample"].eq("test")]
LADDER_FULL = LADDER + [(f"5 {n}", f"pred_{n}") for n in FEATURE_SETS if f"pred_{n}" in ev.columns]
ladder_full = ladder_table(te, LADDER_FULL, REF)
print("full T+1 ladder, target-equal test MAE (bps); reference = cpp_side_eff x beta:")
print(ladder_full.round(3).to_string())
ladder_full.to_csv(OUT_DIR / "ladder_full.csv")

fig, axes = plt.subplots(1, 2, figsize=(17, 6), gridspec_kw={"width_ratios": [3, 2]})
ax = axes[0]
_lf = ladder_full.sort_values("test_mae", ascending=False)
ax.barh(_lf.index, _lf.test_mae, color=["tab:gray" if "dummy" in i else "tab:blue" if "lgbm" in i else "tab:orange" for i in _lf.index], alpha=0.85)
for f in sorted(te.fold_id.unique()):
    ax.scatter(_lf[f"fold{f}"], _lf.index, marker="|", s=120, color="k", label=f"fold {f}" if f == 1 else None)
ax.axvline(ladder_full.loc[ladder_full.index[ladder_full.index.str.startswith('2e')][0], "test_mae"], color="r", ls="--", lw=1, label="cpp_side_eff x beta")
ax.set_xlabel("target-equal test MAE (bps)"); ax.set_title("ladder (bars = all test days; ticks = folds)"); ax.legend(fontsize=8)
ax.set_xlim(_lf.test_mae.min() * 0.9, _lf.test_mae.max() * 1.03)
ax = axes[1]
if RUN_ABLATION and "pred_lgbm-old-core" in ev.columns:
    base_mae = _wmae(te, "pred_lgbm-old-core")
    ab = {k: 100 * (1 - _wmae(te, f"pred_lgbm-old+{k}") / base_mae) for k in BLOCKS if f"pred_lgbm-old+{k}" in ev.columns}
    ab["ALL"] = 100 * (1 - _wmae(te, "pred_lgbm-all") / base_mae)
    ab_fold = {k: [100 * (1 - _wmae(g, f"pred_lgbm-old+{k}" if k != "ALL" else "pred_lgbm-all") / _wmae(g, "pred_lgbm-old-core")) for _, g in te.groupby("fold_id")] for k in ab}
    ax.bar(list(ab.keys()), list(ab.values()), color="tab:green", alpha=0.8)
    for i, k in enumerate(ab):
        ax.scatter([i] * len(ab_fold[k]), ab_fold[k], color="k", marker="_", s=150)
    ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("% MAE gain vs lgbm-old-core"); ax.set_title("block ablation: gain from adding each block to the old core")
    ablation = pd.DataFrame({"gain_%_vs_old_core": ab, **{f"fold{f + 1}": {k: v[f] for k, v in ab_fold.items()} for f in range(len(windows))}})
    for k in ab:
        _, _, ablation.loc[k, "t_vs_old_core"], _ = _paired_days(te, "pred_lgbm-all" if k == "ALL" else f"pred_lgbm-old+{k}", "pred_lgbm-old-core")
    print("\nblock ablation:"); print(ablation.round(3).to_string()); ablation.to_csv(OUT_DIR / "ablation.csv")
savefig("results_5_4_ladder_ablation")

# ---- splits ----------------------------------------------------------------------------------------------------------------------
SPLIT_RUNGS = [("cpp_side", "pred_cpp_side"), ("cpp_side_eff", "pred_cpp_side_eff"), ("cpp_side_eff x beta", "pred_cpp_eff_beta"), ("quality beta", "pred_quality_beta"),
               ("lad-plus", "pred_ladplus")] + [(n, f"pred_{n}") for n in ["lgbm-old-core", "lgbm-all"] if f"pred_{n}" in ev.columns]
def split_table(frame, by):
    rows = []
    for lvl, g in frame.groupby(by, observed=True):
        r = {by: str(lvl), "n": len(g), "share_%": 100 * len(g) / len(frame), "dummy": _wmae(g, "pred_dummy")}
        for name, col in SPLIT_RUNGS:
            r[name] = _wmae(g, col)
        rows.append(r)
    out = pd.DataFrame(rows).set_index(by)
    out["gain_%_eff_vs_tt"] = 100 * (1 - out["cpp_side_eff"] / out["cpp_side"])
    if "lgbm-all" in out.columns and "lgbm-old-core" in out.columns:
        out["gain_%_all_vs_oldcore"] = 100 * (1 - out["lgbm-all"] / out["lgbm-old-core"])
    return out
for by in ["TARGET_CLASS", "PRINT_CLASS", "IS_MISLABELED_D"]:
    print(f"\ntest MAE (bps) by {by}:"); print(split_table(te, by).round(3).to_string())
_mis = te.assign(target_mislabeled=(te.TRADE_TYPE_NEXT.astype(str).eq("D") & te.EFF_SIDE_NEXT.astype(str).ne("D")).map({True: "target D but customer-sided", False: "other"}))
print("\ntest MAE (bps) by whether the TARGET print is a mislabeled D:"); print(split_table(_mis, "target_mislabeled").round(3).to_string())

### 5.5 Residual test: do the new features carry information the old features do not?

The companion's Section 6.6 test, applied to the new blocks only: take the test-fold residual of a reference predictor, fit a LightGBM on the fold's *validation* rows using only the new features, apply it once to the test rows. If MAE falls, the new features carry out-of-sample information that the reference did not use. Two references: `cpp_side_eff x beta` (the simple rule) and `lgbm-old-core` (the old information set).

In [ ]:
if RUN_RESIDUAL_TEST:
    NEW_ALL = [c for c in dict.fromkeys(sum(BLOCKS.values(), []) + PAIR_NEW) if c in pairs.columns and c != "SIDE_PAIR_PRIOR_EFF"]
    _RP = dict(objective="mae", boosting_type="gbdt", learning_rate=0.05, n_estimators=300, num_leaves=31, min_child_samples=500, subsample=0.8, subsample_freq=1,
               colsample_bytree=0.75, reg_lambda=5.0, n_jobs=-1, verbosity=-1)
    key = ["TRADE_ROW_ID", "TRADE_ROW_ID_NEXT"]
    feat = pairs[key + [c for c in NEW_ALL if c not in ev.columns]].drop_duplicates(key)
    rows = []
    for ref_name, ref_col in [("cpp_side_eff x beta", "pred_cpp_eff_beta")] + ([("lgbm-old-core", "pred_lgbm-old-core")] if "pred_lgbm-old-core" in ev.columns else []):
        ev[f"pred_resid_{ref_col}"] = np.nan
        for fold_id, tr_dates, va_dates, te_dates in windows:
            _fill_priors(pairs["DATE"].isin(tr_dates).to_numpy())
            fv = ev[ev.fold_id.eq(fold_id) & ev["sample"].eq("val")].merge(feat, on=key, how="left")
            ft = ev[ev.fold_id.eq(fold_id) & ev["sample"].eq("test")].merge(feat, on=key, how="left")
            if len(fv) > 250_000:
                fv = fv.sample(250_000, random_state=SEED + fold_id)
            r = (fv[ref_col] - fv.y_true).to_numpy(float) * 100
            cats = [c for c in CAT_FEATURES if c in NEW_ALL]
            for c in cats:
                fv[c] = pd.Categorical(fv[c].astype(str), categories=pairs[c].cat.categories)
                ft[c] = pd.Categorical(ft[c].astype(str), categories=pairs[c].cat.categories)
            mdl = lgb.LGBMRegressor(**{**_RP, "random_state": SEED + fold_id, "n_estimators": int(np.clip(len(fv) / 1000, 30, 300))})
            mdl.fit(fv[NEW_ALL], r, categorical_feature=cats)
            corr = mdl.predict(ft[NEW_ALL]) / 100
            em = ev.fold_id.eq(fold_id) & ev["sample"].eq("test")
            ev.loc[em, f"pred_resid_{ref_col}"] = (ft[ref_col].to_numpy(float) - corr)
        t_ = ev[ev["sample"].eq("test")]
        n_, g_, tt_, w_ = _paired_days(t_, f"pred_resid_{ref_col}", ref_col)
        rows.append({"reference": ref_name, "ref_mae": _wmae(t_, ref_col), "mae_after_new_feature_residual_model": _wmae(t_, f"pred_resid_{ref_col}"),
                     "gain_%": 100 * (1 - _wmae(t_, f"pred_resid_{ref_col}") / _wmae(t_, ref_col)), "paired_day_t": tt_, "days_win": w_,
                     "oracle_gap_closed_%": 100 * (_wmae(t_, ref_col) - _wmae(t_, f"pred_resid_{ref_col}")) / (_wmae(t_, ref_col) - bps(t_.TARGET_PRINT_NOISE).abs().mean())})
    resid_test = pd.DataFrame(rows).set_index("reference")
    print("validation-trained residual model on NEW features only, applied to test (companion notebook's 6.6b protocol):")
    print(resid_test.round(3).to_string()); resid_test.to_csv(OUT_DIR / "residual_test.csv")
    print("reading: the companion notebook found -5.4% oracle-gap closed with the OLD features; a positive number here is information the old stack did not have.")

### 5.6 Permutation importance of the full model, aggregated by block

In [ ]:
if RUN_PERMUTATION and "lgbm-all" in LGBM_MODELS:
    feats = FEATURE_SETS["lgbm-all"]
    perm = {}
    for mdl, (fold_id, tr_dates, va_dates, _) in zip(LGBM_MODELS["lgbm-all"], windows):
        _fill_priors(pairs["DATE"].isin(tr_dates).to_numpy())
        rng = np.random.default_rng(SEED + fold_id)
        va_idx = pairs.index[pairs["DATE"].isin(va_dates)]
        idx = rng.choice(va_idx, size=min(PERM_ROWS, len(va_idx)), replace=False)
        X = pairs.loc[idx, feats].reset_index(drop=True); y = pairs.loc[idx, "TARGET"].to_numpy(float)
        base = np.mean(np.abs(mdl.predict(X) - y))
        out = {}
        for f in tqdm(feats, desc=f"permutation fold {fold_id}", leave=False):
            Xp = X.copy(); Xp[f] = Xp[f].sample(frac=1.0, random_state=int(rng.integers(1 << 31))).set_axis(Xp.index)
            out[f] = (np.mean(np.abs(mdl.predict(Xp) - y)) - base) * 100.0
        perm[fold_id] = pd.Series(out)
    perm_df = pd.DataFrame(perm)
    def block_of(f):
        if f in PAIR_NEW or f in SIDE_FEATURES: return "SIDE"
        if f in CPPBIAS_FEATURES: return "CPPBIAS"
        if f in MULTIDAY_FEATURES: return "MULTIDAY"
        if f in TIME_FEATURES: return "TIME"
        return "OLD_CORE"
    fd = pd.DataFrame({"perm_bps": perm_df.mean(axis=1), "perm_min": perm_df.min(axis=1), "perm_std": perm_df.std(axis=1)})
    fd["block"] = [block_of(f) for f in fd.index]
    fd = fd.sort_values("perm_bps", ascending=False)
    print("permutation importance (validation MAE increase in bps when shuffled), top 35:"); print(fd.head(35).round(4).to_string())
    print("\nby block (sum of positive permutation losses):"); print(fd[fd.perm_bps > 0].groupby("block").perm_bps.agg(["sum", "count"]).round(3).to_string())
    fd.to_csv(OUT_DIR / "permutation_importance.csv")
    fig, ax = plt.subplots(figsize=(10, 9))
    _top = fd.head(30).iloc[::-1]
    ax.barh(_top.index, _top.perm_bps, color=_top.block.map({"OLD_CORE": "tab:gray", "SIDE": "tab:red", "CPPBIAS": "tab:blue", "MULTIDAY": "tab:green", "TIME": "tab:purple"}))
    for b, c in {"OLD_CORE": "tab:gray", "SIDE": "tab:red", "CPPBIAS": "tab:blue", "MULTIDAY": "tab:green", "TIME": "tab:purple"}.items():
        ax.scatter([], [], color=c, label=b)
    ax.legend(); ax.set_xlabel("validation MAE increase when shuffled (bps)"); ax.set_title("permutation importance, lgbm-all (top 30)")
    savefig("results_5_6_permutation")

### 5.7 Two-stage model: predict the CPP drift, then add it to `cpp_side_eff`

Section 4 says the whole gap is CPP drift. Instead of learning `TARGET` (drift + print noise), learn `CPP_DRIFT` directly with the same features (much less label noise), then predict `TARGET` as `cpp_side_eff + predicted drift`. The cell reports the out-of-sample correlation between predicted and realised drift, the MAE of the two-stage predictor against the ladder reference, and which features the drift model uses.

In [ ]:
if RUN_DRIFT_MODEL and RUN_LGBM:
    feats = FEATURE_SETS["lgbm-all"]
    drift_models = fit_lgbm("drift", feats, target="CPP_DRIFT", out_col="pred_drift")
    ev["pred_two_stage"] = ev["CPP_TARGET_SIDE_DEV_EFF"] + ev["pred_drift"]
    te = ev[ev["sample"].eq("test")]
    print(f"\ntest corr(predicted drift, realised drift) = {te.pred_drift.corr(te.CPP_DRIFT):+.3f}; "
          f"MAE of drift: dummy {bps(te.CPP_DRIFT).abs().mean():.3f} -> predicted {bps(te.CPP_DRIFT - te.pred_drift).abs().mean():.3f} bps")
    two = ladder_table(te, [("2e cpp_side_eff x beta", "pred_cpp_eff_beta"), ("5 lgbm-all (TARGET)", "pred_lgbm-all"), ("6 two-stage: cpp_side_eff + predicted drift", "pred_two_stage")], REF)
    print(two.round(3).to_string()); two.to_csv(OUT_DIR / "two_stage.csv")
    imp = pd.concat([pd.Series(m.booster_.feature_importance("gain"), index=feats) for m in drift_models], axis=1).mean(axis=1)
    imp = (100 * imp / imp.sum()).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 6)); imp.head(25).iloc[::-1].plot.barh(ax=ax); ax.set_xlabel("gain share %"); ax.set_title("drift model: gain importance (top 25)")
    savefig("results_5_7_drift_model_importance")

## 6. Verdict

The cell below prints a compact readout that can be pasted into the companion notebook's decision table, and saves every table produced above under `artifacts/experiments/next_day_anchor_features/`. The bar the companion used for a dedicated model was "$\ge$5% over the best simple predictor in every fold with $t>2$"; the bar for a *label fix* or a *rule change* should be lower, because it costs no model: positive in every fold with paired-day $t > 2$.

In [ ]:
te = ev[ev["sample"].eq("test")]
def _gain(col, ref):
    n_, g_, t_, w_ = _paired_days(te, col, ref)
    per_fold = [100 * (1 - _wmae(g, col) / _wmae(g, ref)) for _, g in te.groupby("fold_id")]
    return 100 * (1 - _wmae(te, col) / _wmae(te, ref)), per_fold, t_, w_
lines = []
def say(label, col, ref):
    if col not in te.columns or te[col].notna().mean() < 0.5: return
    g, pf, t_, w_ = _gain(col, ref)
    ok = min(pf) > 0 and t_ > 2
    lines.append(f"  {label:<62s} {g:+6.2f}%  folds {np.round(pf, 2).tolist()}  t={t_:5.2f}  wins {w_:.0%}  -> {'PASS' if ok else 'not stable'}")
print(f"test pairs {len(te):,} over {te.DATE.nunique()} days; dummy {_wmae(te, 'pred_dummy'):.3f}, cpp_side {_wmae(te, 'pred_cpp_side'):.3f}, "
      f"cpp_side_eff x beta {_wmae(te, 'pred_cpp_eff_beta'):.3f}, CPP-at-target oracle {bps(te.TARGET_PRINT_NOISE).abs().mean():.3f} bps\n")
say("label hygiene: cpp_side_eff vs cpp_side", "pred_cpp_side_eff", "pred_cpp_side")
say("cpp_side_eff x beta vs cpp_side x beta", "pred_cpp_eff_beta", "pred_cpp_beta")
say("quality beta vs cpp_side_eff x beta", "pred_quality_beta", "pred_cpp_eff_beta")
say("lad-plus vs cpp_side_eff x beta", "pred_ladplus", "pred_cpp_eff_beta")
say("lad-plus vs quality beta", "pred_ladplus", "pred_quality_beta")
if "pred_lgbm-all" in te.columns:
    say("lgbm-all vs lgbm-old-core", "pred_lgbm-all", "pred_lgbm-old-core")
    say("lgbm-all vs cpp_side_eff x beta", "pred_lgbm-all", "pred_cpp_eff_beta")
    say("lgbm-all vs lad-plus", "pred_lgbm-all", "pred_ladplus")
if "pred_two_stage" in te.columns:
    say("two-stage drift model vs cpp_side_eff x beta", "pred_two_stage", "pred_cpp_eff_beta")
    say("two-stage drift model vs lgbm-all", "pred_two_stage", "pred_lgbm-all")
print("\n".join(lines))
_rfq = te[te.TARGET_CLASS.eq("RFQ-like")]
if len(_rfq):
    print(f"\nRFQ-like targets only ({len(_rfq) / len(te):.0%} of pairs): cpp_side {_wmae(_rfq, 'pred_cpp_side'):.3f} | cpp_side_eff x beta {_wmae(_rfq, 'pred_cpp_eff_beta'):.3f} | "
          f"quality beta {_wmae(_rfq, 'pred_quality_beta'):.3f} | lad-plus {_wmae(_rfq, 'pred_ladplus'):.3f}"
          + (f" | lgbm-all {_wmae(_rfq, 'pred_lgbm-all'):.3f}" if "pred_lgbm-all" in te.columns else ""))
ev.drop(columns=[c for c in ev.columns if c.startswith("pred_resid_")]).to_parquet(OUT_DIR / "test_val_predictions.parquet", index=False)
print(f"\nsaved predictions and tables to {OUT_DIR}; total elapsed {(time.time() - t_start) / 60:.1f} min")